In [1]:
from audio_processing import AudioSignal, TimeFeatures, STFTFeatures
import numpy as np
import pandas as pd
import soundfile as sf
import os
import sys

In [2]:
SR = 22050
DURATION = 3.0
N = 2048
H = 512
OUT_DIR = "test_signals"
EPS = 1e-10

os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
# =============================================================================
# SIGNAL GENERATORS
# =============================================================================

def gen_sine(freq=440.0, amp=0.5, duration=DURATION, sr=SR):
    """Pure sine - known RMS = amp / sqrt(2)"""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    return amp * np.sin(2 * np.pi * freq * t)

def gen_am_sine(carrier_freq=440.0, mod_freq=2.0, mod_depth=0.8, amp=0.5, duration=DURATION, sr=SR):
    """AM modulated sine - envelope varies sinusoidally at mod_freq Hz"""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    envelope = amp * (1.0 + mod_depth * np.sin(2 * np.pi * mod_freq * t))
    carrier = np.sin(2 * np.pi * carrier_freq * t)
    return envelope * carrier

def gen_white_noise(amp=0.5, duration=DURATION, sr=SR, seed=42):
    """Gaussian white noise - crest factor typically 4-5x"""
    rng = np.random.default_rng(seed)
    return amp * rng.standard_normal(int(sr * duration))

def gen_click_train(rate_hz=2.0, amp=0.9, duration=DURATION, sr=SR):
    """Periodic impulses - very high crest, sparse energy"""
    sig = np.zeros(int(sr * duration))
    period_samples = int(sr / rate_hz)
    for i in range(0, len(sig), period_samples):
        decay_len = min(64, len(sig) - i)
        decay = amp * np.exp(-np.linspace(0, 5, decay_len))
        sig[i:i+decay_len] = decay
    return sig

def gen_stepped_energy(n_steps=6, duration=DURATION, sr=SR):
    """Staircase amplitude - DR = 20*log10(max_level/min_level)"""
    n_samples = int(sr * duration)
    step_len = n_samples // n_steps
    levels = [0.05, 0.1, 0.2, 0.4, 0.8, 0.5]
    sig = np.zeros(n_samples)
    t = np.linspace(0, duration, n_samples, endpoint=False)
    sine = np.sin(2 * np.pi * 440.0 * t)
    for i, lvl in enumerate(levels):
        start = i * step_len
        end = start + step_len if i < n_steps - 1 else n_samples
        sig[start:end] = lvl * sine[start:end]
    return sig

def gen_silence_padded(inner_amp=0.5, silence_ratio_target=0.4, duration=DURATION, sr=SR):
    """Sine with silence blocks - known silence ratio"""
    n_samples = int(sr * duration)
    n_silence = int(n_samples * silence_ratio_target)
    n_active = n_samples - n_silence
    t = np.linspace(0, duration, n_samples, endpoint=False)
    sig = inner_amp * np.sin(2 * np.pi * 440.0 * t)
    block_size = n_silence // 3
    positions = [
        (0, block_size),
        (n_active // 2, n_active // 2 + block_size),
        (n_samples - block_size, n_samples)
    ]
    for start, end in positions:
        sig[start:end] = 0.0
    return sig

def gen_percussive_train(onset_rate=4.0, attack_ms=2.0, decay_ms=50.0, amp=0.7, duration=DURATION, sr=SR):
    """Periodic percussive hits - known attack/decay envelope"""
    n_samples = int(sr * duration)
    sig = np.zeros(n_samples)
    period_samples = int(sr / onset_rate)
    attack_samples = int(sr * attack_ms / 1000.0)
    decay_samples = int(sr * decay_ms / 1000.0)
    hit_len = attack_samples + decay_samples
    for i in range(0, n_samples, period_samples):
        end = min(i + hit_len, n_samples)
        actual_len = end - i
        atk_end = min(attack_samples, actual_len)
        attack_env = np.linspace(0, 1, atk_end)
        dec_len = actual_len - atk_end
        decay_env = np.exp(-np.linspace(0, 5, dec_len)) if dec_len > 0 else np.array([])
        envelope = np.concatenate([attack_env, decay_env])
        rng = np.random.default_rng(seed=i)
        burst = rng.standard_normal(actual_len)
        sig[i:end] = amp * envelope * burst
    return sig

In [4]:
# =============================================================================
# GROUND TRUTH
# =============================================================================

def compute_ground_truth():
    """
    Expected feature ranges per signal.
    Format: { feature_name: (min_expected, max_expected) }
    """
    gt = {}

    sine_rms = 0.5 / np.sqrt(2)
    gt["sine"] = {
        "rms_median":       (sine_rms * 0.9, sine_rms * 1.1),
        "crest_median":     (1.3, 1.6),
        "dynamic_range":    (0.0, 3.0),
        "energy_variance":  (0.0, 0.15),
        "energy_mod_rate":  (0.0, 0.01),
    }

    gt["am_sine"] = {
        "rms_median":       (0.1, 0.6),
        "crest_median":     (1.3, 2.0),
        "dynamic_range":    (5.0, 20.0),
        "energy_variance":  (0.3, 2.0),
        "energy_mod_rate":  (0.001, 0.01),
    }

    gt["white_noise"] = {
        "rms_median":       (0.45, 0.55),
        "crest_median":     (1.8, 2.5),
        "dynamic_range":    (0.0, 1.0),
        "energy_variance":  (0.0, 0.3),
        "energy_mod_rate":  (0.0, 0.1),
    }

    gt["click_train"] = {
        "rms_median":       (0.0, 0.05),
        "crest_median":     (10.0, 200.0),
        "dynamic_range":    (20.0, 60.0),
        "energy_variance":  (0.0, 0.01),
        "energy_mod_rate":  (0.0, 0.01),
    }

    gt["stepped_energy"] = {
        "rms_median":       (0.1, 0.4),
        "crest_median":     (1.3, 1.6),
        "dynamic_range":    (18.0, 28.0),
        "energy_variance":  (0.5, 3.0),
        "energy_mod_rate":  (0.001, 0.01),
    }

    gt["silence_padded"] = {
        "rms_median":       (0.0, 0.4),
        "crest_median":     (1.3, 50.0),
        "dynamic_range":    (60.0, 80.0),
        "energy_variance":  (0.0, 0.1),
        "energy_mod_rate":  (0.0, 0.02),
    }

    gt["percussive"] = {
        "rms_median":       (0.0, 0.15),
        "crest_median":     (3.0, 30.0),
        "dynamic_range":    (60.0, 80.0),
        "energy_variance":  (0.5, 5.0),
        "energy_mod_rate":  (0.05, 0.8),
        "attack_time":      (0.0, 0.005),
    }

    return gt

In [5]:
# =============================================================================
# GENERATE + SAVE
# =============================================================================

def generate_all_signals():
    """Generate and save all test signals as .wav files."""
    signals = {
        "sine":             gen_sine(),
        "am_sine":          gen_am_sine(),
        "white_noise":      gen_white_noise(),
        "click_train":      gen_click_train(),
        "stepped_energy":   gen_stepped_energy(),
        "silence_padded":   gen_silence_padded(),
        "percussive":       gen_percussive_train(),
    }
    paths = {}
    for name, sig in signals.items():
        path = os.path.join(OUT_DIR, f"{name}.wav")
        sf.write(path, sig.astype(np.float32), SR)
        print(f"  Saved {path} | {len(sig)} samples | {len(sig)/SR:.2f}s")
        paths[name] = path
    return paths

In [6]:
# =============================================================================
# FEATURE EXTRACTION
# =============================================================================

def extract_features(file_paths):
    """
    Instantiate AudioSignal + TimeFeatures for each file.
    Pulls scalar results into a flat DataFrame.
    
    ADAPT: uncomment the import at the top of this file
    and point sys.path to your project directory.
    """
    # --- UNCOMMENT THESE when you have your module on the path ---
    from audio_processing import AudioSignal, TimeFeatures
    
    results = []
    for name, path in file_paths.items():
        print(f"  Processing: {name} ({path})")

        try:
            # --- 1. Load via AudioSignal ---
            sig = AudioSignal(path, N=N, H=H)

            # --- 2. Check validity BEFORE calling features ---
            if sig.invalid:
                print(f"    [SKIP] AudioSignal marked invalid: {name}")
                results.append({
                    'signal': name,
                    'invalid': True,
                    'rms_median': np.nan,
                    'rms_std': np.nan,
                    'crest_median': np.nan,
                    'crest_std': np.nan,
                    'dynamic_range': np.nan,
                    'energy_variance': np.nan,
                    'energy_mod_rate': np.nan,
                    'attack_time': np.nan,
                    'attack_slope': np.nan,
                    'decay_slope': np.nan,
                })
                continue

            # --- 3. Instantiate TimeFeatures ---
            tf = TimeFeatures(sig)

            # --- 4. Extract arrays, aggregate here (not inside the class) ---
            rms = tf._rms_envelope()
            crest = tf._crest_factor()
            mask = tf._active_rms_mask()

            results.append({
                'signal':            name,
                'invalid':           False,
                'rms_median':        float(np.median(rms)),              # Full envelope - silence IS meaningful here
                'rms_std':           float(np.std(rms)),
                'crest_median':      float(np.median(crest[mask])) if mask.any() else 0.0,  # Active only
                'crest_std':         float(np.std(crest[mask])) if mask.any() else 0.0,     # Active only
                'dynamic_range':     float(tf._dynamic_range()),
                'energy_variance':   float(tf.energy_variance()),
                'energy_mod_rate':   float(tf._energy_modulation_rate()),
                'attack_time':       float(tf._attack_time()),
                'attack_slope':      float(tf._attack_slope()),
                'decay_slope':       float(tf._decay_slope()),
            })

        except Exception as e:
            print(f"    [ERROR] {name}: {e}")
            import traceback
            traceback.print_exc()
            results.append({
                'signal': name,
                'invalid': True,
                'rms_median': np.nan,
                'rms_std': np.nan,
                'crest_median': np.nan,
                'crest_std': np.nan,
                'dynamic_range': np.nan,
                'energy_variance': np.nan,
                'energy_mod_rate': np.nan,
                'attack_time': np.nan,
                'attack_slope': np.nan,
                'decay_slope': np.nan,
            })

    return pd.DataFrame(results)

In [7]:
# =============================================================================
# VALIDATION
# =============================================================================

def validate(df, ground_truth):
    """Compare extracted features against ground truth ranges."""
    rows = []
    for _, row in df.iterrows():
        name = row['signal']
        if name not in ground_truth:
            continue
        gt = ground_truth[name]
        for feature, (expected_min, expected_max) in gt.items():
            if feature not in row or pd.isna(row[feature]):
                rows.append({
                    'signal': name,
                    'feature': feature,
                    'actual': None,
                    'expected_min': expected_min,
                    'expected_max': expected_max,
                    'status': 'SKIP (invalid)'
                })
                continue

            actual = row[feature]
            passed = expected_min <= actual <= expected_max
            rows.append({
                'signal': name,
                'feature': feature,
                'actual': round(actual, 6),
                'expected_min': expected_min,
                'expected_max': expected_max,
                'status': 'PASS' if passed else 'FAIL'
            })

    return pd.DataFrame(rows)


def print_report(validation_df):
    """Pretty-print the validation report."""
    total = len(validation_df)
    passed = (validation_df['status'] == 'PASS').sum()
    failed = (validation_df['status'] == 'FAIL').sum()
    skipped = validation_df['status'].str.contains('SKIP').sum()

    print("\n" + "=" * 80)
    print("  AMPLITUDE/ENERGY FEATURE TESTBED REPORT")
    print("=" * 80)
    print(f"  Total: {total} | PASS: {passed} | FAIL: {failed} | SKIPPED: {skipped}")
    print("=" * 80)

    for signal_name in validation_df['signal'].unique():
        subset = validation_df[validation_df['signal'] == signal_name]
        sig_pass = (subset['status'] == 'PASS').sum()
        sig_total = len(subset)
        icon = "✅" if sig_pass == sig_total else "❌"

        print(f"\n  {icon} {signal_name} ({sig_pass}/{sig_total} passed)")
        print(f"  {'─' * 74}")
        print(f"  {'Feature':<25} {'Actual':>12} {'Expected Range':>22} {'Status':>10}")
        print(f"  {'─' * 74}")

        for _, r in subset.iterrows():
            if r['status'] == 'PASS':
                s_icon = "✅ PASS"
            elif 'SKIP' in str(r['status']):
                s_icon = "⚠️  SKIP"
            else:
                s_icon = "❌ FAIL"

            actual_str = f"{r['actual']:.6f}" if r['actual'] is not None else "N/A"
            range_str = f"[{r['expected_min']:.4f}, {r['expected_max']:.4f}]"
            print(f"  {r['feature']:<25} {actual_str:>12} {range_str:>22} {s_icon:>10}")

    print("\n" + "=" * 80)

In [8]:
print("\n[1/4] Generating test signals...")
file_paths = generate_all_signals()


[1/4] Generating test signals...
  Saved test_signals\sine.wav | 66150 samples | 3.00s
  Saved test_signals\am_sine.wav | 66150 samples | 3.00s
  Saved test_signals\white_noise.wav | 66150 samples | 3.00s
  Saved test_signals\click_train.wav | 66150 samples | 3.00s
  Saved test_signals\stepped_energy.wav | 66150 samples | 3.00s
  Saved test_signals\silence_padded.wav | 66150 samples | 3.00s
  Saved test_signals\percussive.wav | 66150 samples | 3.00s


In [9]:
print("\n[2/4] Extracting features...")
df = extract_features(file_paths)
print("\n  Raw results:")
print(df.to_string(index=False))


[2/4] Extracting features...
  Processing: sine (test_signals\sine.wav)
    [ERROR] sine: 'TimeFeatures' object has no attribute 'energy_variance'
  Processing: am_sine (test_signals\am_sine.wav)
    [ERROR] am_sine: 'TimeFeatures' object has no attribute 'energy_variance'
  Processing: white_noise (test_signals\white_noise.wav)
    [ERROR] white_noise: 'TimeFeatures' object has no attribute 'energy_variance'
  Processing: click_train (test_signals\click_train.wav)
    [ERROR] click_train: 'TimeFeatures' object has no attribute 'energy_variance'
  Processing: stepped_energy (test_signals\stepped_energy.wav)
    [ERROR] stepped_energy: 'TimeFeatures' object has no attribute 'energy_variance'
  Processing: silence_padded (test_signals\silence_padded.wav)
    [ERROR] silence_padded: 'TimeFeatures' object has no attribute 'energy_variance'
  Processing: percussive (test_signals\percussive.wav)
    [ERROR] percussive: 'TimeFeatures' object has no attribute 'energy_variance'

  Raw results:

Traceback (most recent call last):
  File "C:\Users\mythk\AppData\Local\Temp\ipykernel_38848\3330302857.py", line 59, in extract_features
    'energy_variance':   float(tf.energy_variance()),
                               ^^^^^^^^^^^^^^^^^^
AttributeError: 'TimeFeatures' object has no attribute 'energy_variance'
Traceback (most recent call last):
  File "C:\Users\mythk\AppData\Local\Temp\ipykernel_38848\3330302857.py", line 59, in extract_features
    'energy_variance':   float(tf.energy_variance()),
                               ^^^^^^^^^^^^^^^^^^
AttributeError: 'TimeFeatures' object has no attribute 'energy_variance'
Traceback (most recent call last):
  File "C:\Users\mythk\AppData\Local\Temp\ipykernel_38848\3330302857.py", line 59, in extract_features
    'energy_variance':   float(tf.energy_variance()),
                               ^^^^^^^^^^^^^^^^^^
AttributeError: 'TimeFeatures' object has no attribute 'energy_variance'
Traceback (most recent call last):
  File "C:\Users\myt

In [10]:
print("\n[3/4] Validating against ground truth...")
gt = compute_ground_truth()
validation = validate(df, gt)


[3/4] Validating against ground truth...


In [11]:
print("\n[4/4] Report:")
print_report(validation)


[4/4] Report:

  AMPLITUDE/ENERGY FEATURE TESTBED REPORT
  Total: 36 | PASS: 0 | FAIL: 0 | SKIPPED: 36

  ❌ sine (0/5 passed)
  ──────────────────────────────────────────────────────────────────────────
  Feature                         Actual         Expected Range     Status
  ──────────────────────────────────────────────────────────────────────────
  rms_median                         N/A       [0.3182, 0.3889]   ⚠️  SKIP
  crest_median                       N/A       [1.3000, 1.6000]   ⚠️  SKIP
  dynamic_range                      N/A       [0.0000, 3.0000]   ⚠️  SKIP
  energy_variance                    N/A       [0.0000, 0.1500]   ⚠️  SKIP
  energy_mod_rate                    N/A       [0.0000, 0.0100]   ⚠️  SKIP

  ❌ am_sine (0/5 passed)
  ──────────────────────────────────────────────────────────────────────────
  Feature                         Actual         Expected Range     Status
  ──────────────────────────────────────────────────────────────────────────
  rms_median  

In [12]:
# Add this temporarily to extract_features():
import librosa
sig = AudioSignal("test_signals/sine.wav", N=N, H=H)
onset_env = librosa.onset.onset_strength(y=sig.y, sr=sig.sr, hop_length=H)
onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sig.sr, hop_length=H, units='frames')
print(f"  sine: onset_env max={onset_env.max():.4f}, num_onsets={len(onsets)}")

  sine: onset_env max=0.0486, num_onsets=1


In [13]:
# In extract_features(), for click_train only:
sig = AudioSignal("test_signals/click_train.wav", N=N, H=H)
tf = TimeFeatures(sig)
rms = tf._rms_envelope()
peak = tf._peak_amplitude()
mask = tf._active_rms_mask()

rms_db = 20.0 * np.log10(rms + 1e-10)
peak_db = 20.0 * np.log10(peak + 1e-10)

print(f"  Total frames:  {len(mask)}")
print(f"  Active frames: {mask.sum()}")
print(f"  RMS  dB range: [{rms_db.min():.1f}, {rms_db.max():.1f}]")
print(f"  Peak dB range: [{peak_db.min():.1f}, {peak_db.max():.1f}]")
print(f"  Frames where peak_db > -60: {(peak_db > -60).sum()}")
print(f"  Frames where rms_db  > -60: {(rms_db  > -60).sum()}")

  Total frames:  126
  Active frames: 21
  RMS  dB range: [-200.0, -25.7]
  Peak dB range: [-200.0, -0.9]
  Frames where peak_db > -60: 21
  Frames where rms_db  > -60: 21


In [14]:
SR = 22050
DURATION = 3.0
N = 2048
H = 512
OUT_DIR = "dataset/test_signals_noise"
EPS = 1e-10

os.makedirs(OUT_DIR, exist_ok=True)

In [15]:
# =============================================================================
# SIGNAL GENERATORS
# =============================================================================

def gen_low_freq_sine(freq=100.0, amp=0.5, duration=DURATION, sr=SR):
    """
    Low frequency sine - low ZCR, high periodicity (voiced).
    Expected ZCR = 2*freq/sr = 2*100/22050 ≈ 0.009
    """
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    return amp * np.sin(2 * np.pi * freq * t)

def gen_high_freq_sine(freq=5000.0, amp=0.5, duration=DURATION, sr=SR):
    """
    High frequency sine - high ZCR, still periodic (voiced).
    Expected ZCR = 2*5000/22050 ≈ 0.45
    """
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    return amp * np.sin(2 * np.pi * freq * t)

def gen_white_noise(amp=0.5, duration=DURATION, sr=SR, seed=42):
    """
    White noise - high ZCR (~0.5), no periodicity (unvoiced).
    Expected ZCR ≈ 0.5 (crosses zero ~50% of samples)
    """
    rng = np.random.default_rng(seed)
    return amp * rng.standard_normal(int(sr * duration))

def gen_bandpass_noise(flow=300.0, fhigh=3400.0, amp=0.5, duration=DURATION, sr=SR, seed=42):
    """
    Bandpass filtered noise (telephone band) - speech-like spectrum.
    Expected: moderate ZCR (0.2-0.4), low periodicity
    """
    from scipy.signal import butter, filtfilt
    
    rng = np.random.default_rng(seed)
    noise = rng.standard_normal(int(sr * duration))
    
    # Butterworth bandpass
    nyq = sr / 2.0
    low = flow / nyq
    high = fhigh / nyq
    b, a = butter(4, [low, high], btype='band')
    filtered = filtfilt(b, a, noise)
    
    # Normalize
    filtered = amp * filtered / (np.max(np.abs(filtered)) + EPS)
    return filtered

def gen_pulse_train(freq=10.0, duty_cycle=0.1, amp=0.5, duration=DURATION, sr=SR):
    """Square wave with amplitude envelope to create transients"""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    square = np.sign(np.sin(2 * np.pi * freq * t))
    
    # Add sharp envelope at each transition to create energy transients
    period_samples = int(sr / freq)
    envelope = np.ones_like(square)
    
    for i in range(0, len(square), period_samples // 2):  # Each half-period
        # Sharp attack
        attack_len = min(100, len(envelope) - i)
        envelope[i:i+attack_len] = np.linspace(0.1, 1.0, attack_len)
    
    return amp * square * envelope

def gen_voiced_speech_proxy(f0=150.0, formants=[800, 1200, 2500], amp=0.5, duration=DURATION, sr=SR):
    """
    Voiced speech proxy - sum of harmonics with formant envelope.
    Expected: low ZCR, very high periodicity (VUR ≈ 1.0)
    """
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    
    # Fundamental + harmonics
    sig = np.zeros_like(t)
    for h in range(1, 11):  # 10 harmonics
        harmonic_freq = h * f0
        if harmonic_freq < sr / 2:
            sig += (1.0 / h) * np.sin(2 * np.pi * harmonic_freq * t)
    
    # Formant-like amplitude envelope (simple gaussian bumps)
    envelope = np.ones_like(t)
    for formant in formants:
        envelope += 2.0 * np.exp(-((t % 1.0) - 0.5)**2 / 0.1)
    
    sig = sig * envelope
    sig = amp * sig / (np.max(np.abs(sig)) + EPS)
    return sig

def gen_unvoiced_fricative_proxy(center_freq=4000.0, bandwidth=2000.0, amp=0.5, duration=DURATION, sr=SR, seed=42):
    """
    Unvoiced fricative proxy - highpass filtered noise.
    Expected: very high ZCR (0.4-0.5), no periodicity (VUR ≈ 0.0)
    """
    from scipy.signal import butter, filtfilt
    
    rng = np.random.default_rng(seed)
    noise = rng.standard_normal(int(sr * duration))
    
    # Highpass filter
    nyq = sr / 2.0
    cutoff = center_freq / nyq
    b, a = butter(4, cutoff, btype='high')
    filtered = filtfilt(b, a, noise)
    
    filtered = amp * filtered / (np.max(np.abs(filtered)) + EPS)
    return filtered

def gen_percussive_burst_train(rate=4.0, amp=0.7, duration=DURATION, sr=SR, seed=42):
    """
    Periodic noise bursts - clear transients.
    Expected transient rate = rate (e.g., 4 Hz)
    """
    sig = np.zeros(int(sr * duration))
    period_samples = int(sr / rate)
    burst_len = int(sr * 0.05)  # 50ms bursts
    
    rng = np.random.default_rng(seed)
    
    for i in range(0, len(sig), period_samples):
        burst = amp * rng.standard_normal(burst_len)
        # Exponential envelope
        env = np.exp(-np.linspace(0, 3, burst_len))
        burst = burst * env
        
        end = min(i + burst_len, len(sig))
        sig[i:end] = burst[:end-i]
    
    return sig

def gen_alternating_voiced_unvoiced(segment_duration=0.5, f0=200.0, amp=0.5, duration=DURATION, sr=SR, seed=42):
    """
    Alternating voiced (sine) and unvoiced (noise) segments.
    Expected: moderate VUR (≈ 0.5), high ZCR variance
    """
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    sig = np.zeros_like(t)
    
    segment_samples = int(segment_duration * sr)
    rng = np.random.default_rng(seed)
    
    voiced = True
    for i in range(0, len(sig), segment_samples):
        end = min(i + segment_samples, len(sig))
        
        if voiced:
            # Voiced segment (sine)
            seg_t = t[i:end] - t[i]
            sig[i:end] = amp * np.sin(2 * np.pi * f0 * seg_t)
        else:
            # Unvoiced segment (noise)
            sig[i:end] = amp * rng.standard_normal(end - i)
        
        voiced = not voiced
    
    return sig

In [16]:
# =============================================================================
# GROUND TRUTH
# =============================================================================

def compute_ground_truth():
    """
    Expected feature ranges per signal.
    
    ZCR formula: 2 * freq / sr (for sine waves)
    VUR: 0 = fully unvoiced, 1 = fully voiced
    ZCR Variance: normalized variability
    Transient Rate: onsets per second
    """
    gt = {}

    # --- Low Frequency Sine (100 Hz) ---
    # ZCR = 2*100/22050 ≈ 0.009
    gt["low_freq_sine"] = {
        "zcr_median":       (0.005, 0.015),
        "zcr_variance":     (0.0, 0.1),      # Very stable
        "voiced_ratio":     (0.9, 1.0),      # Highly periodic
        "unvoiced_ratio":   (0.0, 0.1),
        "transient_rate":   (0.0, 0.5),      # No transients
    }

    # --- High Frequency Sine (5000 Hz) ---
    # ZCR = 2*5000/22050 ≈ 0.453
    gt["high_freq_sine"] = {
        "zcr_median":       (0.40, 0.50),
        "zcr_variance":     (0.0, 0.1),      # Stable
        "voiced_ratio":     (0.8, 1.0),      # Still periodic despite high ZCR
        "unvoiced_ratio":   (0.0, 0.2),
        "transient_rate":   (0.0, 0.5),
    }

    # --- White Noise ---
    # ZCR ≈ 0.5 (theoretical), no periodicity
    gt["white_noise"] = {
        "zcr_median":       (0.45, 0.55),
        "zcr_variance":     (0.0, 0.15),     # Low variance (always noisy)
        "voiced_ratio":     (0.0, 0.1),      # Not periodic
        "unvoiced_ratio":   (0.9, 1.0),
        "transient_rate":   (0.0, 2.0),      # Few false detections
    }

    # --- Bandpass Noise (300-3400 Hz) ---
    gt["bandpass_noise"] = {
        "zcr_median":       (0.15, 0.35),
        "zcr_variance":     (0.0, 0.2),
        "voiced_ratio":     (0.0, 0.2),      # Noisy, not periodic
        "unvoiced_ratio":   (0.8, 1.0),
        "transient_rate":   (0.0, 3.0),
    }

    # --- Pulse Train (10 Hz, 10% duty cycle) ---
    # Many zero crossings per period
    gt["pulse_train"] = {
        "zcr_median":       (0.0005, 0.002),    # Depends on duty cycle
        "zcr_variance":     (0.0, 0.3),
        "voiced_ratio":     (0.7, 1.0),      # Periodic but many crossings
        "unvoiced_ratio":   (0.0, 0.3),
        "transient_rate":   (6.0, 12.0),     # ~10 Hz transients
    }

    # --- Voiced Speech Proxy (150 Hz fundamental) ---
    gt["voiced_speech"] = {
        "zcr_median":       (0.01, 0.05),    # Low freq harmonics dominate
        "zcr_variance":     (0.0, 0.2),
        "voiced_ratio":     (0.85, 1.0),     # Highly periodic
        "unvoiced_ratio":   (0.0, 0.15),
        "transient_rate":   (0.0, 1.0),
    }

    # --- Unvoiced Fricative Proxy ---
    gt["unvoiced_fricative"] = {
        "zcr_median":       (0.55, 0.75),    # High freq noise
        "zcr_variance":     (0.0, 0.2),
        "voiced_ratio":     (0.0, 0.1),      # No periodicity
        "unvoiced_ratio":   (0.9, 1.0),
        "transient_rate":   (0.0, 4.0),
    }

    # --- Percussive Burst Train (4 Hz) ---
    gt["percussive_bursts"] = {
        "zcr_median":       (0.03, 0.15),    # Noise bursts
        "zcr_variance":     (0.1, 1.0),      # High variance (bursts vs silence)
        "voiced_ratio":     (0.0, 0.2),      # Not periodic
        "unvoiced_ratio":   (0.8, 1.0),
        "transient_rate":   (3.0, 5.0),      # ~4 Hz
    }

    # --- Alternating Voiced/Unvoiced ---
    gt["alternating"] = {
        "zcr_median":       (0.10, 0.35),    # Mix of both
        "zcr_variance":     (0.3, 2.0),      # High variance (alternates)
        "voiced_ratio":     (0.4, 0.7),      # Mix
        "unvoiced_ratio":   (0.3, 0.6),
        "transient_rate":   (0.0, 4.0),      # Transitions
    }

    return gt

In [17]:
# =============================================================================
# GENERATE + SAVE
# =============================================================================

def generate_all_signals():
    """Generate and save all test signals as .wav files."""
    signals = {
        "low_freq_sine":        gen_low_freq_sine(),
        "high_freq_sine":       gen_high_freq_sine(),
        "white_noise":          gen_white_noise(),
        "bandpass_noise":       gen_bandpass_noise(),
        "pulse_train":          gen_pulse_train(),
        "voiced_speech":        gen_voiced_speech_proxy(),
        "unvoiced_fricative":   gen_unvoiced_fricative_proxy(),
        "percussive_bursts":    gen_percussive_burst_train(),
        "alternating":          gen_alternating_voiced_unvoiced(),
    }
    
    paths = {}
    for name, sig in signals.items():
        path = os.path.join(OUT_DIR, f"{name}.wav")
        sf.write(path, sig.astype(np.float32), SR)
        print(f"  Saved {path} | {len(sig)} samples | {len(sig)/SR:.2f}s")
        paths[name] = path
    
    return paths

In [18]:
# =============================================================================
# FEATURE EXTRACTION
# =============================================================================

def extract_features(file_paths):
    """
    Extract noise/speech features from each test signal.
    
    ADAPT: uncomment import and point to your module.
    """
    # from your_module import AudioSignal, TimeFeatures

    results = []

    for name, path in file_paths.items():
        print(f"  Processing: {name}")

        try:
            sig = AudioSignal(path, N=N, H=H)

            if sig.invalid:
                print(f"    [SKIP] Invalid signal: {name}")
                results.append({
                    'signal': name,
                    'invalid': True,
                    'zcr_median': np.nan,
                    'zcr_variance': np.nan,
                    'voiced_ratio': np.nan,
                    'unvoiced_ratio': np.nan,
                    'transient_rate': np.nan,
                    'transient_count': np.nan,
                })
                continue

            tf = TimeFeatures(sig)

            # Extract features
            zcr = tf._zero_crossing_rate()
            mask = tf._active_rms_mask()

            results.append({
                'signal':           name,
                'invalid':          False,
                'zcr_median':       float(np.median(zcr)),
                'zcr_variance':     float(tf._zcr_variance()),
                'voiced_ratio':     float(tf._voiced_ratio()),
                'unvoiced_ratio':   float(tf._unvoiced_ratio()),
                'transient_rate':   float(tf._transient_rate()),
                'transient_count':  int(tf._transient_counts()),
            })

        except Exception as e:
            print(f"    [ERROR] {name}: {e}")
            import traceback
            traceback.print_exc()
            results.append({
                'signal': name,
                'invalid': True,
                'zcr_median': np.nan,
                'zcr_variance': np.nan,
                'voiced_ratio': np.nan,
                'unvoiced_ratio': np.nan,
                'transient_rate': np.nan,
                'transient_count': np.nan,
            })

    return pd.DataFrame(results)

In [19]:
# =============================================================================
# VALIDATION
# =============================================================================

def validate(df, ground_truth):
    """Compare extracted features against ground truth ranges."""
    rows = []
    for _, row in df.iterrows():
        name = row['signal']
        if name not in ground_truth:
            continue
        
        gt = ground_truth[name]
        for feature, (expected_min, expected_max) in gt.items():
            if feature not in row or pd.isna(row[feature]):
                rows.append({
                    'signal': name,
                    'feature': feature,
                    'actual': None,
                    'expected_min': expected_min,
                    'expected_max': expected_max,
                    'status': 'SKIP (invalid)'
                })
                continue

            actual = row[feature]
            passed = expected_min <= actual <= expected_max
            rows.append({
                'signal': name,
                'feature': feature,
                'actual': round(actual, 6),
                'expected_min': expected_min,
                'expected_max': expected_max,
                'status': 'PASS' if passed else 'FAIL'
            })

    return pd.DataFrame(rows)


def print_report(validation_df):
    """Pretty-print the validation report."""
    total = len(validation_df)
    passed = (validation_df['status'] == 'PASS').sum()
    failed = (validation_df['status'] == 'FAIL').sum()
    skipped = validation_df['status'].str.contains('SKIP').sum()

    print("\n" + "=" * 80)
    print("  NOISE/SPEECH INDICATORS TESTBED REPORT")
    print("=" * 80)
    print(f"  Total: {total} | PASS: {passed} | FAIL: {failed} | SKIPPED: {skipped}")
    print("=" * 80)

    for signal_name in validation_df['signal'].unique():
        subset = validation_df[validation_df['signal'] == signal_name]
        sig_pass = (subset['status'] == 'PASS').sum()
        sig_total = len(subset)
        icon = "✅" if sig_pass == sig_total else "❌"

        print(f"\n  {icon} {signal_name} ({sig_pass}/{sig_total} passed)")
        print(f"  {'─' * 74}")
        print(f"  {'Feature':<25} {'Actual':>12} {'Expected Range':>22} {'Status':>10}")
        print(f"  {'─' * 74}")

        for _, r in subset.iterrows():
            if r['status'] == 'PASS':
                s_icon = "✅ PASS"
            elif 'SKIP' in str(r['status']):
                s_icon = "⚠️  SKIP"
            else:
                s_icon = "❌ FAIL"

            actual_str = f"{r['actual']:.6f}" if r['actual'] is not None else "N/A"
            range_str = f"[{r['expected_min']:.4f}, {r['expected_max']:.4f}]"
            print(f"  {r['feature']:<25} {actual_str:>12} {range_str:>22} {s_icon:>10}")

    print("\n" + "=" * 80)

In [20]:
print("\n[1/4] Generating test signals...")
file_paths = generate_all_signals()


[1/4] Generating test signals...
  Saved dataset/test_signals_noise\low_freq_sine.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_noise\high_freq_sine.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_noise\white_noise.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_noise\bandpass_noise.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_noise\pulse_train.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_noise\voiced_speech.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_noise\unvoiced_fricative.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_noise\percussive_bursts.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_noise\alternating.wav | 66150 samples | 3.00s


In [21]:
print("\n[2/4] Extracting features...")
df = extract_features(file_paths)
print("\n  Raw results:")
print(df.to_string(index=False))


[2/4] Extracting features...
  Processing: low_freq_sine
  Processing: high_freq_sine
  Processing: white_noise
  Processing: bandpass_noise
  Processing: pulse_train
  Processing: voiced_speech
  Processing: unvoiced_fricative
  Processing: percussive_bursts
  Processing: alternating

  Raw results:
            signal  invalid  zcr_median  zcr_variance  voiced_ratio  unvoiced_ratio  transient_rate  transient_count
     low_freq_sine    False    0.009277      0.052632      1.000000    7.936984e-13        0.000000                0
    high_freq_sine    False    0.453125      0.001078      1.000000    7.936984e-13        0.000000                0
       white_noise    False    0.497314      0.033382      0.000000    1.000000e+00        1.333333                4
    bandpass_noise    False    0.176758      0.037293      0.000000    1.000000e+00        1.333333                4
       pulse_train    False    0.000977      0.000000      1.000000    7.936984e-13        9.333333             

In [22]:
print("\n[3/4] Validating against ground truth...")
gt = compute_ground_truth()
validation = validate(df, gt)


[3/4] Validating against ground truth...


In [23]:
print("\n[4/4] Report:")
print_report(validation)


[4/4] Report:

  NOISE/SPEECH INDICATORS TESTBED REPORT
  Total: 45 | PASS: 45 | FAIL: 0 | SKIPPED: 0

  ✅ low_freq_sine (5/5 passed)
  ──────────────────────────────────────────────────────────────────────────
  Feature                         Actual         Expected Range     Status
  ──────────────────────────────────────────────────────────────────────────
  zcr_median                    0.009277       [0.0050, 0.0150]     ✅ PASS
  zcr_variance                  0.052632       [0.0000, 0.1000]     ✅ PASS
  voiced_ratio                  1.000000       [0.9000, 1.0000]     ✅ PASS
  unvoiced_ratio                0.000000       [0.0000, 0.1000]     ✅ PASS
  transient_rate                0.000000       [0.0000, 0.5000]     ✅ PASS

  ✅ high_freq_sine (5/5 passed)
  ──────────────────────────────────────────────────────────────────────────
  Feature                         Actual         Expected Range     Status
  ──────────────────────────────────────────────────────────────────────────

In [24]:
# Add to testbed temporarily:
sig = AudioSignal("dataset/test_signals_noise/pulse_train.wav", N=N, H=H)
print(f"pulse_train: min={sig.y.min():.4f} max={sig.y.max():.4f} mean={sig.y.mean():.4f}")
print(f"  unique values: {np.unique(sig.y)}")

pulse_train: min=-0.5000 max=0.5000 mean=-0.0001
  unique values: [-0.5        -0.4954834  -0.49093628 -0.48638916 -0.48449707 -0.48184204
 -0.47729492 -0.4727478  -0.46899414 -0.46820068 -0.46365356 -0.45910645
 -0.45455933 -0.4534607  -0.4500122  -0.4454651  -0.44091797 -0.43795776
 -0.43637085 -0.43182373 -0.4272766  -0.4227295  -0.42242432 -0.41818237
 -0.41366577 -0.40911865 -0.4069214  -0.40457153 -0.4000244  -0.3954773
 -0.39138794 -0.39093018 -0.38638306 -0.38183594 -0.37728882 -0.375885
 -0.3727417  -0.36819458 -0.36364746 -0.36035156 -0.35910034 -0.35455322
 -0.3500061  -0.34545898 -0.34484863 -0.34091187 -0.33636475 -0.33184814
 -0.3293152  -0.32730103 -0.3227539  -0.3182068  -0.31381226 -0.31365967
 -0.30911255 -0.30456543 -0.3000183  -0.2982788  -0.2954712  -0.29092407
 -0.28637695 -0.28277588 -0.28182983 -0.2772827  -0.2727356  -0.26818848
 -0.26724243 -0.26364136 -0.25909424 -0.25454712 -0.2517395  -0.25
 -0.2454834  -0.24093628 -0.23638916 -0.23623657 -0.23184204 -0.227

In [25]:
SR = 22050
DURATION = 10.0
N = 2048
H = 512
OUT_DIR = "dataset/test_signals_rhythm"
EPS = 1e-10

os.makedirs(OUT_DIR, exist_ok=True)

In [26]:
# =============================================================================
# SIGNAL GENERATORS
# =============================================================================

def gen_metronomic_clicks(bpm=120.0, amp=0.7, duration=DURATION, sr=SR):
    """
    Perfect metronomic click train.
    
    Known properties:
    - Tempo = bpm (exact)
    - IOI = 60/bpm seconds (constant)
    - CV(IOI) = 0 (perfect regularity)
    - Pulse clarity = very high (>0.8)
    - Stability = 1.0 (no tempo variation)
    """
    sig = np.zeros(int(sr * duration))
    period_sec = 60.0 / bpm
    period_samples = int(sr * period_sec)
    
    # Place clicks
    for i in range(0, len(sig), period_samples):
        # Short impulse with exponential decay
        click_len = min(100, len(sig) - i)
        click = amp * np.exp(-np.linspace(0, 5, click_len))
        sig[i:i+click_len] = click
    
    return sig

def gen_accelerando(bpm_start=80.0, bpm_end=160.0, amp=0.7, duration=DURATION, sr=SR):
    """
    Gradual tempo increase (accelerando).
    
    Known properties:
    - Starting tempo ≈ bpm_start
    - Ending tempo ≈ bpm_end
    - IOI decreases linearly
    - CV(IOI) > 0.3 (high variance)
    - Stability < 0.5 (low - tempo changes)
    """
    sig = np.zeros(int(sr * duration))
    
    t = 0.0  # Current time in seconds
    current_bpm = bpm_start
    bpm_rate = (bpm_end - bpm_start) / duration  # BPM change per second
    
    while t < duration:
        # Place click
        sample_idx = int(t * sr)
        if sample_idx >= len(sig):
            break
        
        click_len = min(100, len(sig) - sample_idx)
        click = amp * np.exp(-np.linspace(0, 5, click_len))
        sig[sample_idx:sample_idx+click_len] = click
        
        # Calculate next IOI based on current tempo
        ioi = 60.0 / current_bpm
        t += ioi
        
        # Update tempo
        current_bpm += bpm_rate * ioi
    
    return sig

def gen_irregular_rhythm(avg_ioi=0.5, ioi_std=0.2, amp=0.7, duration=DURATION, sr=SR, seed=42):
    """
    Random IOIs with Gaussian distribution (rubato / free time).
    
    Known properties:
    - Mean IOI ≈ avg_ioi
    - Std(IOI) ≈ ioi_std
    - CV(IOI) = ioi_std / avg_ioi
    - Pulse clarity < 0.3 (weak/no pulse)
    - Stability < 0.5 (irregular)
    """
    rng = np.random.default_rng(seed)
    sig = np.zeros(int(sr * duration))
    
    t = 0.0
    while t < duration:
        # Place click
        sample_idx = int(t * sr)
        if sample_idx >= len(sig):
            break
        
        click_len = min(100, len(sig) - sample_idx)
        click = amp * np.exp(-np.linspace(0, 5, click_len))
        sig[sample_idx:sample_idx+click_len] = click
        
        # Random IOI (clipped to positive)
        ioi = max(0.1, rng.normal(avg_ioi, ioi_std))
        t += ioi
    
    return sig

def gen_polyrhythm_3_over_2(bpm_base=120.0, amp=0.7, duration=12.0, sr=SR):
    """
    3:2 polyrhythm (3 beats against 2 beats in same time span).
    
    Known properties:
    - Two IOI modes: 60/(bpm_base) and 60/(bpm_base*1.5)
    - Pulse clarity medium (0.4-0.7) - multiple periodicities
    - Beat periodicity < 0.7 (bimodal IOI distribution)
    """
    sig = np.zeros(int(sr * duration))
    
    # Stream 1: Base tempo
    period_1 = int(sr * 60.0 / bpm_base)
    for i in range(0, len(sig), period_1):
        click_len = min(100, len(sig) - i)
        click = amp * np.exp(-np.linspace(0, 5, click_len))
        sig[i:i+click_len] += click * 0.7  # Slightly quieter
    
    # Stream 2: 1.5x faster
    period_2 = int(sr * 60.0 / (bpm_base * 1.5))
    for i in range(0, len(sig), period_2):
        click_len = min(100, len(sig) - i)
        click = amp * np.exp(-np.linspace(0, 5, click_len))
        sig[i:i+click_len] += click * 0.5  # Even quieter
    
    # Normalize
    sig = np.clip(sig, -1.0, 1.0)
    return sig

def gen_syncopated_pattern(bpm=120.0, amp=0.7, duration=DURATION, sr=SR):
    """
    Syncopated rhythm (strong offbeats, weak downbeats).
    Pattern: weak-STRONG-weak-STRONG per measure
    
    Known properties:
    - Tempo = bpm
    - IOI = constant (60/bpm)
    - Pulse clarity medium-high (0.5-0.8) - clear pulse but accents vary
    - Stability high (>0.8) - tempo is stable
    """
    sig = np.zeros(int(sr * duration))
    period_sec = 60.0 / bpm
    period_samples = int(sr * period_sec)
    
    # Pattern: weak, strong, weak, strong (4 beats per measure)
    pattern = [0.3, 0.9, 0.4, 0.9]
    
    beat_idx = 0
    for i in range(0, len(sig), period_samples):
        # Accent based on pattern
        accent = pattern[beat_idx % len(pattern)]
        
        click_len = min(100, len(sig) - i)
        click = amp * accent * np.exp(-np.linspace(0, 5, click_len))
        sig[i:i+click_len] = click
        
        beat_idx += 1
    
    return sig

def gen_swing_rhythm(bpm=120.0, swing_ratio=2.0, amp=0.7, duration=DURATION, sr=SR):
    """
    Swing/shuffle rhythm (long-short pattern).
    
    Known properties:
    - Average tempo ≈ bpm
    - IOI alternates: long (swing_ratio × base) and short
    - CV(IOI) moderate (0.2-0.4)
    - Pulse clarity high (0.7-0.9) - strong periodic pattern
    """
    sig = np.zeros(int(sr * duration))
    
    # Base eighth note duration
    base_ioi = 60.0 / (bpm * 2)  # bpm is quarter notes, we want eighths
    
    # Swing: first note is longer, second is shorter
    # Ratio of 2:1 means triplet feel (2/3 + 1/3 of beat)
    total_time = base_ioi * 2
    ioi_long = total_time * (swing_ratio / (swing_ratio + 1))
    ioi_short = total_time * (1 / (swing_ratio + 1))
    
    t = 0.0
    is_long = True
    
    while t < duration:
        sample_idx = int(t * sr)
        if sample_idx >= len(sig):
            break
        
        click_len = min(100, len(sig) - sample_idx)
        click = amp * np.exp(-np.linspace(0, 5, click_len))
        sig[sample_idx:sample_idx+click_len] = click
        
        # Alternate long/short
        t += ioi_long if is_long else ioi_short
        is_long = not is_long
    
    return sig

def gen_ritardando(bpm_start=140.0, bpm_end=70.0, amp=0.7, duration=DURATION, sr=SR):
    """
    Gradual slowdown (ritardando).
    
    Known properties:
    - Starting tempo ≈ bpm_start
    - Ending tempo ≈ bpm_end
    - IOI increases over time
    - Stability < 0.5
    """
    sig = np.zeros(int(sr * duration))
    
    t = 0.0
    current_bpm = bpm_start
    bpm_rate = (bpm_end - bpm_start) / duration
    
    while t < duration:
        sample_idx = int(t * sr)
        if sample_idx >= len(sig):
            break
        
        click_len = min(100, len(sig) - sample_idx)
        click = amp * np.exp(-np.linspace(0, 5, click_len))
        sig[sample_idx:sample_idx+click_len] = click
        
        ioi = 60.0 / current_bpm
        t += ioi
        current_bpm += bpm_rate * ioi
    
    return sig

def gen_no_rhythm(amp=0.5, duration=DURATION, sr=SR, seed=42):
    """
    Continuous noise with no rhythmic structure.
    
    Known properties:
    - No clear tempo
    - Few/no onsets (depends on energy threshold)
    - Pulse clarity ≈ 0
    - All rhythm metrics should be near-zero or undefined
    """
    rng = np.random.default_rng(seed)
    return amp * rng.standard_normal(int(sr * duration))

In [27]:
# =============================================================================
# GROUND TRUTH
# =============================================================================

def compute_ground_truth():
    """
    Expected feature ranges per signal.
    
    Key metrics:
    - onset_rate: onsets/second
    - tempo: BPM from autocorrelation
    - ioi_mean: average inter-onset-interval (seconds)
    - ioi_cv: coefficient of variation (std/mean)
    - pulse_clarity: 0-1 (AC peak dominance)
    - stability_exp: 0-1 (tempo consistency)
    - periodicity: 0-1 (IOI entropy-based)
    """
    gt = {}

    # --- Metronomic Clicks (120 BPM) ---
    # IOI = 60/120 = 0.5 sec
    # Onset rate = 120/60 = 2 Hz
    gt["metronomic_120"] = {
        "onset_rate":       (1.8, 2.2),      # ~2 Hz
        "tempo":            (115.0, 125.0),  # 120 BPM
        "ioi_mean":         (0.48, 0.52),    # 0.5 sec
        "ioi_cv":           (0.0, 0.05),     # Nearly zero variance
        "pulse_clarity":    (0.75, 1.0),     # Very high
        "stability_exp":    (0.95, 1.0),     # Perfect stability
        "periodicity":      (0.7, 1.0),     # Highly periodic
    }

    # --- Accelerando (80 → 160 BPM) ---
    # Average tempo ≈ 120, but high variance
    gt["accelerando"] = {
        "onset_rate":       (1.5, 3.0),      # Increases over time
        "tempo":            (140.0, 180.0),  # Global estimate varies
        "ioi_mean":         (0.4, 0.7),      # Average IOI
        "ioi_cv":           (0.15, 0.50),     # High variance
        "pulse_clarity":    (0.05, 0.5),      # Moderate (tempo change hurts AC)
        "stability_exp":    (0.0, 0.7),      # Low (tempo changes)
        "periodicity":      (0.15, 0.8),      # Moderate
    }

    # --- Irregular Rhythm (avg_ioi=0.5, std=0.2) ---
    # CV = 0.2/0.5 = 0.4
    gt["irregular"] = {
        "onset_rate":       (1.5, 2.5),      # ~2 Hz average
        "tempo":            (80.0, 160.0),   # Weak/unreliable
        "ioi_mean":         (0.4, 0.6),      # 0.5 target
        "ioi_cv":           (0.3, 0.5),      # 0.4 target
        "pulse_clarity":    (0.0, 0.3),      # Very weak
        "stability_exp":    (0.0, 0.3),      # Low
        "periodicity":      (0.0, 0.4),      # Low
    }

    # --- Polyrhythm 3:2 (base 120 BPM) ---
    # Two IOI modes: 0.5 sec (120 BPM) and 0.333 sec (180 BPM)
    gt["polyrhythm"] = {
        "onset_rate":       (3.0, 5.5),      # More onsets (two streams)
        "tempo":            (100.0, 140.0),  # May lock to either stream
        "ioi_mean":         (0.20, 0.30),     # Mix of both periods
        "ioi_cv":           (0.2, 0.5),      # Bimodal distribution
        "pulse_clarity":    (0.7, 1.00),      # Moderate (competing pulses)
        "stability_exp":    (0.6, 1.0),     # Stable (both streams regular)
        "periodicity":      (0.0, 1.0),      # Lower (bimodal IOI)
    }

    # --- Syncopated (120 BPM) ---
    # Same IOI as metronomic, just different accents
    gt["syncopated"] = {
        "onset_rate":       (1.8, 2.2),      # 2 Hz
        "tempo":            (115.0, 125.0),  # 120 BPM
        "ioi_mean":         (0.48, 0.52),    # 0.5 sec
        "ioi_cv":           (0.0, 0.05),     # Regular timing
        "pulse_clarity":    (0.5, 1.00),     # High (accents don't destroy pulse)
        "stability_exp":    (0.9, 1.0),      # Very stable
        "periodicity":      (0.7, 1.0),      # Highly periodic
    }

    # --- Swing Rhythm (120 BPM, 2:1 ratio) ---
    # IOIs alternate ~0.4 and ~0.2 sec
    gt["swing"] = {
        "onset_rate":       (3.5, 4.5),      # ~4 Hz (eighth notes)
        "tempo":            (115.0, 125.0),  # 120 BPM (quarter note)
        "ioi_mean":         (0.22, 0.28),    # Average of long/short
        "ioi_cv":           (0.25, 0.45),    # Moderate (alternating pattern)
        "pulse_clarity":    (0.65, 0.95),    # High (strong periodic pattern)
        "stability_exp":    (0.85, 1.0),     # Stable
        "periodicity":      (0.55, 0.9),      # High but bimodal IOI
    }

    # --- Ritardando (140 → 70 BPM) ---
    gt["ritardando"] = {
        "onset_rate":       (1.2, 2.5),      # Decreases over time
        "tempo":            (110.0, 140.0),   # Global estimate
        "ioi_mean":         (0.5, 0.9),      # Increases
        "ioi_cv":           (0.15, 0.6),      # High variance
        "pulse_clarity":    (0.05, 0.50),      # Moderate
        "stability_exp":    (0.0, 1.0),      # Low
        "periodicity":      (0.15, 0.8),      # Moderate
    }

    # --- No Rhythm (noise) ---
    gt["no_rhythm"] = {
        "onset_rate":       (0.0, 5.0),      # Few/no onsets
        "tempo":            (0.0, 240.0),    # Meaningless
        "ioi_mean":         (0.0, 5.0),      # Undefined/unreliable
        "ioi_cv":           (0.0, 2.0),      # Undefined
        "pulse_clarity":    (0.0, 0.2),      # No pulse
        "stability_exp":    (0.0, 0.5),      # Undefined
        "periodicity":      (0.0, 0.3),      # No structure
    }

    return gt

In [28]:
# =============================================================================
# GENERATE + SAVE
# =============================================================================

def generate_all_signals():
    """Generate and save all test signals."""
    signals = {
        "metronomic_120":   gen_metronomic_clicks(bpm=120.0),
        "accelerando":      gen_accelerando(bpm_start=80.0, bpm_end=160.0),
        "irregular":        gen_irregular_rhythm(avg_ioi=0.5, ioi_std=0.2),
        "polyrhythm":       gen_polyrhythm_3_over_2(bpm_base=120.0),
        "syncopated":       gen_syncopated_pattern(bpm=120.0),
        "swing":            gen_swing_rhythm(bpm=120.0, swing_ratio=2.0),
        "ritardando":       gen_ritardando(bpm_start=140.0, bpm_end=70.0),
        "no_rhythm":        gen_no_rhythm(),
    }
    
    paths = {}
    for name, sig in signals.items():
        path = os.path.join(OUT_DIR, f"{name}.wav")
        sf.write(path, sig.astype(np.float32), SR)
        print(f"  Saved {path} | {len(sig)} samples | {len(sig)/SR:.2f}s")
        paths[name] = path
    
    return paths

In [29]:
# =============================================================================
# FEATURE EXTRACTION
# =============================================================================

def extract_features(file_paths):
    """
    Extract rhythm/beat features from each test signal.
    
    ADAPT: uncomment import and point to your module.
    """
    # from your_module import AudioSignal, TimeFeatures

    results = []

    for name, path in file_paths.items():
        print(f"  Processing: {name}")

        try:
            sig = AudioSignal(path, N=N, H=H)

            if sig.invalid:
                print(f"    [SKIP] Invalid signal: {name}")
                results.append({
                    'signal': name,
                    'invalid': True,
                    **{k: np.nan for k in ['onset_rate', 'tempo', 'ioi_mean', 
                                            'ioi_cv', 'pulse_clarity', 
                                            'stability_exp', 'periodicity']}
                })
                continue

            tf = TimeFeatures(sig)

            # Add to extract_features() for debugging:
            if name in ["metronomic_120", "irregular", "accelerando", "no_rhythm"]:
                ac = tf._onset_autocorrelation()
                print(f"\n  {name} AC diagnostic:")
                print(f"    AC[0]={ac[0]:.6f}")
                print(f"    AC[1:11]={ac[1:11]}")
                print(f"    AC max(1:)={np.max(ac[1:]):.6f}, mean(1:)={np.mean(ac[1:]):.6f}")
                
                tempos = tf._windowed_tempo_series()
                print(f"    Windowed tempos: {tempos}")
                print(f"    Tempo std: {np.std(tempos):.4f}")

            # Extract features
            ioi_mean, ioi_std, ioi_cv = tf._ioi_stats()
            stability_dict = tf._rhythmic_stability()

            results.append({
                'signal':           name,
                'invalid':          False,
                'onset_rate':       float(tf._onset_rate()),
                'tempo':            float(tf._tempo_from_onset_ac()),
                'ioi_mean':         float(ioi_mean),
                'ioi_cv':           float(ioi_cv),
                'pulse_clarity':    float(tf._pulse_clarity_ac()),
                'stability_exp':    float(stability_dict['stability_exp']),
                'periodicity':      float(tf._beat_periodicity_entropy()),
            })

        except Exception as e:
            print(f"    [ERROR] {name}: {e}")
            import traceback
            traceback.print_exc()
            results.append({
                'signal': name,
                'invalid': True,
                **{k: np.nan for k in ['onset_rate', 'tempo', 'ioi_mean', 
                                        'ioi_cv', 'pulse_clarity', 
                                        'stability_exp', 'periodicity']}
            })

    return pd.DataFrame(results)

In [30]:
# =============================================================================
# VALIDATION
# =============================================================================

def validate(df, ground_truth):
    """Compare extracted features against ground truth ranges."""
    rows = []
    for _, row in df.iterrows():
        name = row['signal']
        if name not in ground_truth:
            continue
        
        gt = ground_truth[name]
        for feature, (expected_min, expected_max) in gt.items():
            if feature not in row or pd.isna(row[feature]):
                rows.append({
                    'signal': name,
                    'feature': feature,
                    'actual': None,
                    'expected_min': expected_min,
                    'expected_max': expected_max,
                    'status': 'SKIP (invalid)'
                })
                continue

            actual = row[feature]
            passed = expected_min <= actual <= expected_max
            rows.append({
                'signal': name,
                'feature': feature,
                'actual': round(actual, 6),
                'expected_min': expected_min,
                'expected_max': expected_max,
                'status': 'PASS' if passed else 'FAIL'
            })

    return pd.DataFrame(rows)


def print_report(validation_df):
    """Pretty-print the validation report."""
    total = len(validation_df)
    passed = (validation_df['status'] == 'PASS').sum()
    failed = (validation_df['status'] == 'FAIL').sum()
    skipped = validation_df['status'].str.contains('SKIP').sum()

    print("\n" + "=" * 80)
    print("  RHYTHM/BEAT FEATURE TESTBED REPORT")
    print("=" * 80)
    print(f"  Total: {total} | PASS: {passed} | FAIL: {failed} | SKIPPED: {skipped}")
    print("=" * 80)

    for signal_name in validation_df['signal'].unique():
        subset = validation_df[validation_df['signal'] == signal_name]
        sig_pass = (subset['status'] == 'PASS').sum()
        sig_total = len(subset)
        icon = "✅" if sig_pass == sig_total else "❌"

        print(f"\n  {icon} {signal_name} ({sig_pass}/{sig_total} passed)")
        print(f"  {'─' * 74}")
        print(f"  {'Feature':<25} {'Actual':>12} {'Expected Range':>22} {'Status':>10}")
        print(f"  {'─' * 74}")

        for _, r in subset.iterrows():
            if r['status'] == 'PASS':
                s_icon = "✅ PASS"
            elif 'SKIP' in str(r['status']):
                s_icon = "⚠️  SKIP"
            else:
                s_icon = "❌ FAIL"

            actual_str = f"{r['actual']:.6f}" if r['actual'] is not None else "N/A"
            range_str = f"[{r['expected_min']:.2f}, {r['expected_max']:.2f}]"
            print(f"  {r['feature']:<25} {actual_str:>12} {range_str:>22} {s_icon:>10}")

    print("\n" + "=" * 80)

In [31]:
print("\n[1/4] Generating test signals...")
file_paths = generate_all_signals()


[1/4] Generating test signals...
  Saved dataset/test_signals_rhythm\metronomic_120.wav | 220500 samples | 10.00s
  Saved dataset/test_signals_rhythm\accelerando.wav | 220500 samples | 10.00s
  Saved dataset/test_signals_rhythm\irregular.wav | 220500 samples | 10.00s
  Saved dataset/test_signals_rhythm\polyrhythm.wav | 264600 samples | 12.00s
  Saved dataset/test_signals_rhythm\syncopated.wav | 220500 samples | 10.00s
  Saved dataset/test_signals_rhythm\swing.wav | 220500 samples | 10.00s
  Saved dataset/test_signals_rhythm\ritardando.wav | 220500 samples | 10.00s
  Saved dataset/test_signals_rhythm\no_rhythm.wav | 220500 samples | 10.00s


In [32]:
print("\n[2/4] Extracting features...")
df = extract_features(file_paths)
print("\n  Raw results:")
print(df.to_string(index=False))


[2/4] Extracting features...
  Processing: metronomic_120

  metronomic_120 AC diagnostic:
    AC[0]=1.000000
    AC[1:11]=[ 0.21377888 -0.06142894 -0.07434491 -0.07451621 -0.07468751 -0.07485881
 -0.07503012 -0.07520141 -0.07537272 -0.07554402]
    AC max(1:)=0.876985, mean(1:)=-0.001163
    Windowed tempos: [117.45383523 117.45383523 117.45383523]
    Tempo std: 0.0000
  Processing: accelerando

  accelerando AC diagnostic:
    AC[0]=1.000000
    AC[1:11]=[ 0.16728118 -0.06329283 -0.06883483 -0.06899343 -0.06915203 -0.06931064
 -0.06946925 -0.06962785 -0.06937096 -0.0663236 ]
    AC max(1:)=0.167281, mean(1:)=-0.001163
    Windowed tempos: [129.19921875 151.99908088 151.99908088]
    Tempo std: 10.7480
  Processing: irregular

  irregular AC diagnostic:
    AC[0]=1.000000
    AC[1:11]=[ 0.25251568 -0.06306981 -0.08396249 -0.06774172 -0.01499858 -0.07161193
 -0.08053927 -0.08073273 -0.07241797 -0.01740324]
    AC max(1:)=0.252516, mean(1:)=-0.001163
    Windowed tempos: [129.19921875

In [33]:
print("\n[3/4] Validating against ground truth...")
gt = compute_ground_truth()
validation = validate(df, gt)


[3/4] Validating against ground truth...


In [34]:
print("\n[4/4] Report:")
print_report(validation)


[4/4] Report:

  RHYTHM/BEAT FEATURE TESTBED REPORT
  Total: 56 | PASS: 56 | FAIL: 0 | SKIPPED: 0

  ✅ metronomic_120 (7/7 passed)
  ──────────────────────────────────────────────────────────────────────────
  Feature                         Actual         Expected Range     Status
  ──────────────────────────────────────────────────────────────────────────
  onset_rate                    1.900000           [1.80, 2.20]     ✅ PASS
  tempo                       117.453835       [115.00, 125.00]     ✅ PASS
  ioi_mean                      0.500519           [0.48, 0.52]     ✅ PASS
  ioi_cv                        0.023721           [0.00, 0.05]     ✅ PASS
  pulse_clarity                 0.901588           [0.75, 1.00]     ✅ PASS
  stability_exp                 1.000000           [0.95, 1.00]     ✅ PASS
  periodicity                   0.770687           [0.70, 1.00]     ✅ PASS

  ✅ accelerando (7/7 passed)
  ──────────────────────────────────────────────────────────────────────────
  Featu

In [35]:
SR = 22050
DURATION = 10.0
N = 2048
H = 512
OUT_DIR = "dataset/test_signals_correlation"
EPS = 1e-10

os.makedirs(OUT_DIR, exist_ok=True)

In [36]:
# ══════════════════════════════════════════════════════════════════════════════
#  Signal generators  (all analytically tractable)
# ══════════════════════════════════════════════════════════════════════════════

def make_dc_signal(duration=2.0, amplitude=1.0):
    """Constant signal — mean-centering zeroes it; AC is undefined / trivially 0."""
    n = int(duration * SR)
    return np.full(n, amplitude)


def make_pure_sine(freq=440.0, duration=2.0, amplitude=1.0):
    """
    Single-frequency sine.
    AC of a sine A·sin(2πft) is (A²/2)·cos(2πfτ):
      • AC[0]  = 1.0  (normalized)
      • Peak lags at multiples of T = sr/freq samples
      • lag-1 correlation ≈ cos(2π·freq/sr)  ≈ 1 for low freq
    """
    t = np.arange(int(duration * SR)) / SR
    return amplitude * np.sin(2 * np.pi * freq * t)


def make_white_noise(duration=2.0, seed=42):
    """
    White noise — AC → delta function:
      • AC[0] = 1.0
      • AC[k≠0] ≈ 0  (within sqrt(1/N) noise)
      • lag-1, lag-2 ≈ 0
    """
    rng = np.random.default_rng(seed)
    return rng.standard_normal(int(duration * SR))


def make_periodic_impulse(period_samples=1000, duration=2.0):
    """
    Impulse train — exact periodicity:
      • Strong AC peaks at lags k·period_samples
      • lag-1 ≈ 0 (adjacent samples mostly zero)
    """
    n = int(duration * SR)
    y = np.zeros(n)
    y[::period_samples] = 1.0
    return y


def make_repeated_segment(segment_len=4096, repeats=8):
    """
    Exactly repeated random block:
      • Self-similarity matrix should be ~1.0 at diagonal offsets = k·segment_len/H
      • AC has strong peak at lag = segment_len
    """
    rng = np.random.default_rng(7)
    seg = rng.standard_normal(segment_len)
    return np.tile(seg, repeats)


def make_two_alternating_segments(seg_len=4096, repeats=4):
    """
    Alternating A B A B … segments:
      • SSM: high similarity at even offsets (A-A, B-B), low at odd (A-B)
      • AC: peaks at even multiples of seg_len
    """
    rng = np.random.default_rng(13)
    seg_a = rng.standard_normal(seg_len)
    seg_b = rng.standard_normal(seg_len)
    pieces = []
    for i in range(repeats * 2):
        pieces.append(seg_a if i % 2 == 0 else seg_b)
    return np.concatenate(pieces)


def make_linear_ramp(duration=2.0):
    """
    Linear ramp 0→1 — after mean-centering it's a centered ramp.
    lag-1 correlation of a ramp is very close to +1.
    """
    n = int(duration * SR)
    return np.linspace(0.0, 1.0, n)


def make_ar1_process(phi=0.95, duration=2.0, seed=42, burn_in=5000):
    """
    AR(1): x[t] = phi·x[t-1] + ε
    Discard burn_in samples to reach stationary distribution.
    """
    rng = np.random.default_rng(seed)
    n_total = int(duration * SR) + burn_in
    x = np.zeros(n_total)
    for i in range(1, n_total):
        x[i] = phi * x[i - 1] + rng.standard_normal()
    
    # Discard burn-in, return stationary part
    return x[burn_in:]


def make_constant_frames(num_frames=10):
    """
    Build a signal whose every frame (length N, hop H) contains the same
    waveform.  The key: place the sine segment so every hop-aligned window
    of length N is identical.  The only way to guarantee this with hop < N
    is to make the entire signal one repeating sine — then every frame is
    a different phase-slice of the same sinusoid, which after mean-centering
    and L2-normalisation has cosine similarity = cos(Δφ) ≈ 1 for low freq.

    Instead, use hop == N (non-overlapping) so frames are exact copies.
    """
    t   = np.arange(N) / SR
    seg = np.sin(2 * np.pi * 440 * t)   # one frame of 440 Hz sine
    # Non-overlapping: total length = num_frames * N
    # We'll override H with N when constructing this feature object.
    return np.tile(seg, num_frames)

def make_short_signal():
    """Short signal for testing frame extraction edge case - must be non-silent"""
    n = N // 2
    t = np.arange(n) / SR
    # Tiny amplitude to avoid silence detection
    return 0.001 * np.sin(2 * np.pi * 440 * t)

In [37]:
# ══════════════════════════════════════════════════════════════════════════════
#  Ground-truth derivations
# ══════════════════════════════════════════════════════════════════════════════

def expected_sine_lag1(freq, sr=SR):
    """Theoretical lag-1 AC of a pure sine = cos(2π·freq/sr)."""
    return float(np.cos(2 * np.pi * freq / sr))


def expected_ar1_lagk(phi, k):
    """Theoretical lag-k AC of AR(1) = phi^k."""
    return float(phi ** k)


def expected_frame_count(signal_len, frame_len=N, hop=H):
    """Number of frames produced by _frames()."""
    if signal_len < frame_len:
        return 0
    return 1 + (signal_len - frame_len) // hop

In [38]:
# ══════════════════════════════════════════════════════════════════════════════
#  Test runner
# ══════════════════════════════════════════════════════════════════════════════

PASS_MARK = "✅ PASS"
FAIL_MARK = "❌ FAIL"
WIDTH     = 74


def check(name, actual, lo, hi):
    ok = lo <= actual <= hi
    status = PASS_MARK if ok else FAIL_MARK
    print(f"  {name:<32} {actual:>12.6f}   [{lo:.4f}, {hi:.4f}]   {status}")
    return ok


def section(title):
    print(f"\n  {'─'*WIDTH}")
    print(f"  {title}")
    print(f"  {'─'*WIDTH}")
    print(f"  {'Feature':<32} {'Actual':>12}   {'Expected Range':<20}   Status")
    print(f"  {'─'*WIDTH}")


# ── tolerance helpers ────────────────────────────────────────────────────────
def tol(center, rel=0.05, abs_=0.01):
    """Return (lo, hi) = center ± max(rel*|center|, abs_)."""
    margin = max(rel * abs(center), abs_)
    return center - margin, center + margin

In [39]:
# ══════════════════════════════════════════════════════════════════════════════
#  Generate all signals
# ══════════════════════════════════════════════════════════════════════════════
def gen_all_signals():
    signals = {
        "dc_signal": make_dc_signal(),
        "pure_signal_low": make_pure_sine(freq=110.0, duration=2.0),
        "pure_signal_high": make_pure_sine(freq=4410.0, duration=1.0),
        "pure_signal_frames": make_pure_sine(freq=220.0, duration=2.0),
        "white_noise": make_white_noise(duration=3.0),
        "periodic_impulse": make_periodic_impulse(period_samples=1000, duration=3.0),
        "repeated_segment": make_repeated_segment(segment_len=4096, repeats=6),
        "two_alternating_segments": make_two_alternating_segments(seg_len=4096, repeats=4),
        "linear_ramp": make_linear_ramp(duration=2.0),
        "ar1_process": make_ar1_process(phi=0.95, duration=4.0),
        "ssm_white_noise": np.random.default_rng(99).standard_normal(41472),
        "signal_short": make_short_signal(),
        "constant_frames": make_constant_frames(num_frames=10)
    }

    paths = {}
    for name, sig in signals.items():
        path = os.path.join(OUT_DIR, f"{name}.wav")
        sf.write(path, sig.astype(np.float32), SR)
        print(f"  Saved {path} | {len(sig)} samples | {len(sig)/SR:.2f}s")
        paths[name] = path
    
    return paths

In [40]:
paths = gen_all_signals()

  Saved dataset/test_signals_correlation\dc_signal.wav | 44100 samples | 2.00s
  Saved dataset/test_signals_correlation\pure_signal_low.wav | 44100 samples | 2.00s
  Saved dataset/test_signals_correlation\pure_signal_high.wav | 22050 samples | 1.00s
  Saved dataset/test_signals_correlation\pure_signal_frames.wav | 44100 samples | 2.00s
  Saved dataset/test_signals_correlation\white_noise.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_correlation\periodic_impulse.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_correlation\repeated_segment.wav | 24576 samples | 1.11s
  Saved dataset/test_signals_correlation\two_alternating_segments.wav | 32768 samples | 1.49s
  Saved dataset/test_signals_correlation\linear_ramp.wav | 44100 samples | 2.00s
  Saved dataset/test_signals_correlation\ar1_process.wav | 88200 samples | 4.00s
  Saved dataset/test_signals_correlation\ssm_white_noise.wav | 41472 samples | 1.88s
  Saved dataset/test_signals_correlation\signal_short.wav | 1024 sam

In [41]:
# ══════════════════════════════════════════════════════════════════════════════
#  Individual test suites
# ══════════════════════════════════════════════════════════════════════════════

results = {}   # signal_name -> (passed, total)


def run_suite(name, signal, tests):
    passed = sum(tests)
    total  = len(tests)
    results[name] = (passed, total)
    icon = "✅" if passed == total else "❌"
    print(f"\n  {icon} {name} ({passed}/{total} passed)")


# ─────────────────────────────────────────────────────────────────────────────
#  1. DC / zero-variance signal
# ─────────────────────────────────────────────────────────────────────────────
section("DC signal  (constant value — AC is degenerate)")
y_dc = paths["dc_signal"]
feat_dc = AudioSignal(audio_path=y_dc)
tf = TimeFeatures(feat_dc)
ac_dc = tf._autocorrelation()

# After mean-centering a DC signal the whole array is 0 → ac[0]=0 → no normalization
# The implementation returns ac/ac[0] only if ac[0]>0, so ac stays all-zero.
t = []
t.append(check("ac_length_equals_signal",   len(ac_dc),      len(feat_dc.y)-1, len(feat_dc.y)+1))
t.append(check("ac_zero_due_to_dc",          float(np.max(np.abs(ac_dc))), 0.0, EPS*10))
r1_dc = tf._lag_k_correlation(1)
t.append(check("lag1_dc_signal",             r1_dc,          -EPS, EPS))
run_suite("dc_signal", y_dc, t)


# ─────────────────────────────────────────────────────────────────────────────
#  2. Pure sine — 110 Hz (low frequency, long period)
# ─────────────────────────────────────────────────────────────────────────────
section("Pure sine 110 Hz")
FREQ_LOW = 110.0
y_sine_low = paths["pure_signal_low"]
feat_sl = AudioSignal(audio_path=y_sine_low)
tf = TimeFeatures(feat_sl)

ac_sl       = tf._autocorrelation()
period_samp = SR / FREQ_LOW          # = 200.45… samples
r1_theory   = expected_sine_lag1(FREQ_LOW)   # ≈ 0.99997

t = []
t.append(check("ac[0]_normalized",           ac_sl[0],       1.0-EPS, 1.0+EPS))

# AC should peak at lag ≈ period_samples
peaks_sl = tf._autocorrelation_peaks(min_lag=1)
if peaks_sl["lags"].size > 0:
    first_peak_lag = peaks_sl["lags"][0]
    t.append(check("first_peak_lag_near_period",
                   float(first_peak_lag),
                   period_samp * 0.90, period_samp * 1.10))
    t.append(check("first_peak_value_high",
                   peaks_sl["values"][0],  0.85, 1.0))
else:
    t += [False, False]
    print("  !! No AC peaks found for 110 Hz sine")

r1_sl = tf._lag_k_correlation(1)
lo1, hi1 = tol(r1_theory, rel=0.001, abs_=0.002)
t.append(check("lag1_matches_theory",        r1_sl,          lo1, hi1))

r2_sl = tf._lag_k_correlation(2)
r2_th = expected_sine_lag1(FREQ_LOW * 2)   # cos(4π·freq/sr) but that's lag-2 of sine
# lag-2 of sine = cos(2·2π·freq/sr) = cos(4π·freq/sr)
r2_theory = float(np.cos(4 * np.pi * FREQ_LOW / SR))
lo2, hi2  = tol(r2_theory, rel=0.001, abs_=0.002)
t.append(check("lag2_matches_theory",        r2_sl,          lo2, hi2))

run_suite("sine_110hz", y_sine_low, t)


# ─────────────────────────────────────────────────────────────────────────────
#  3. Pure sine — 4410 Hz (high frequency, short period → lag-1 ≈ cos(π/5))
# ─────────────────────────────────────────────────────────────────────────────
section("Pure sine 4410 Hz  (period = 5 samples)")
FREQ_HIGH = 4410.0
y_sine_high = paths["pure_signal_high"]
feat_sh = AudioSignal(audio_path=y_sine_high)
tf = TimeFeatures(feat_sh)

r1_theory_h = expected_sine_lag1(FREQ_HIGH)   # cos(2π·4410/22050)=cos(2π/5)≈0.309
ac_sh = tf._autocorrelation()
t = []
t.append(check("ac[0]_normalized",           ac_sh[0],       1.0-EPS, 1.0+EPS))

r1_sh = tf._lag_k_correlation(1)
lo, hi = tol(r1_theory_h, rel=0.01, abs_=0.01)
t.append(check("lag1_cos_2pi_over_5",        r1_sh,          lo, hi))

# AC peak at lag=5 (period in samples)
peaks_sh = tf._autocorrelation_peaks(min_lag=1)
if peaks_sh["lags"].size > 0:
    min_lag_found = peaks_sh["lags"][0]
    t.append(check("first_ac_peak_lag_5",    float(min_lag_found), 4.0, 6.0))
else:
    t.append(False)
    print("  !! No AC peaks for 4410 Hz sine")

run_suite("sine_4410hz", y_sine_high, t)


# ─────────────────────────────────────────────────────────────────────────────
#  4. White noise — AC should be near-delta
# ─────────────────────────────────────────────────────────────────────────────
section("White noise  (AC ≈ delta, lag-k ≈ 0)")
y_wn   = paths["white_noise"]
feat_wn = AudioSignal(audio_path=y_wn)
tf = TimeFeatures(feat_wn)

ac_wn  = tf._autocorrelation()

# Theoretical std of AC coefficients for white noise = 1/sqrt(N)
N_wn      = len(y_wn)
ac_std_th = 1.0 / np.sqrt(N_wn)
tail_std  = float(np.std(ac_wn[1:1000]))

t = []
t.append(check("ac[0]_normalized",           ac_wn[0],       1.0-EPS, 1.0+EPS))
t.append(check("tail_std_near_1_over_sqrtN", tail_std,       0.0, 5*ac_std_th))

r12 = tf._lag1_lag2_correlations()
t.append(check("lag1_near_zero",             r12["lag1"],   -0.03, 0.03))
t.append(check("lag2_near_zero",             r12["lag2"],   -0.03, 0.03))

# Peaks exist but none should be strongly dominant
peaks_wn = tf._autocorrelation_peaks(min_lag=1)
if peaks_wn["values"].size > 0:
    max_peak_wn = float(peaks_wn["values"].max())
    t.append(check("max_ac_peak_weak",       max_peak_wn,    0.0, 0.15))
else:
    t.append(True)   # no peaks at all is also acceptable

run_suite("white_noise", y_wn, t)


# ─────────────────────────────────────────────────────────────────────────────
#  5. Periodic impulse train
# ─────────────────────────────────────────────────────────────────────────────
section("Periodic impulse train  (period = 1000 samples)")
PERIOD = 1000
y_imp   = paths["periodic_impulse"]
feat_imp = AudioSignal(audio_path=y_imp)
tf = TimeFeatures(feat_imp)
ac_imp  = tf._autocorrelation()

t = []
t.append(check("ac[0]_normalized",           ac_imp[0],      1.0-EPS, 1.0+EPS))

peaks_imp = tf._autocorrelation_peaks(min_lag=1)
if peaks_imp["lags"].size > 0:
    # A sparse impulse train generates many tiny wiggles in the AC.
    # Use the STRONGEST peak — analytically it must sit at lag = PERIOD.
    best_idx = int(np.argmax(peaks_imp["values"]))
    best_lag = float(peaks_imp["lags"][best_idx])
    best_val = float(peaks_imp["values"][best_idx])
    t.append(check("strongest_peak_at_period",  best_lag, PERIOD*0.95, PERIOD*1.05))
    t.append(check("strongest_peak_value_high", best_val, 0.85, 1.0))
    # Harmonic: strongest peak in [1.8, 2.2] × PERIOD
    hmask = ((peaks_imp["lags"] >= PERIOD * 1.80) &
             (peaks_imp["lags"] <= PERIOD * 2.20))
    if hmask.any():
        h_val = float(peaks_imp["values"][hmask].max())
        t.append(check("harmonic_2period_exists", h_val, 0.50, 1.0))
    else:
        t.append(True)   # harmonic optional if signal too short
else:
    t += [False, False, False]
    print("  !! No peaks found for impulse train")

# lag-1 of impulse train ≈ 0 (adjacent samples are 0)
r1_imp = tf._lag_k_correlation(1)
t.append(check("lag1_near_zero",             r1_imp,        -0.02, 0.02))

run_suite("impulse_train_1000", y_imp, t)


# ─────────────────────────────────────────────────────────────────────────────
#  6. AR(1) process — lag-k = phi^k
# ─────────────────────────────────────────────────────────────────────────────
section("AR(1) process  phi=0.95  (lag-k theory = phi^k)")
PHI = 0.95
y_ar  = make_ar1_process(phi=0.95, duration=4.0)
feat_ar = AudioSignal(signal=y_ar, N=N, H=H)
tf = TimeFeatures(feat_ar)

t = []
for k in [1, 2, 5, 10]:
    theory = expected_ar1_lagk(PHI, k)
    actual = tf._lag_k_correlation(k)
    lo, hi = tol(theory, rel=0.05, abs_=0.03)
    t.append(check(f"lag{k}_vs_phi^{k}",    actual,          lo, hi))

run_suite("ar1_phi095", y_ar, t)


# ─────────────────────────────────────────────────────────────────────────────
#  7. Linear ramp — lag-1 near +1
# ─────────────────────────────────────────────────────────────────────────────
section("Linear ramp  (lag-1 ≈ +1.0)")
y_ramp   = paths["linear_ramp"]
feat_ramp = AudioSignal(audio_path=y_ramp)
tf = TimeFeatures(feat_ramp)

t = []
r1_ramp = tf._lag_k_correlation(1)
t.append(check("lag1_near_plus1",            r1_ramp,        0.999, 1.0))
r2_ramp = tf._lag_k_correlation(2)
t.append(check("lag2_near_plus1",            r2_ramp,        0.999, 1.0))

# AC of a ramp: after mean-centering it's still monotone → peak at lag=1
ac_ramp  = tf._autocorrelation()
t.append(check("ac[0]_normalized",           ac_ramp[0],     1.0-EPS, 1.0+EPS))
t.append(check("ac_monotone_decreasing",
               float(ac_ramp[1] > ac_ramp[100]),  # ac[1] > ac[100]
               1.0-EPS, 1.0+EPS))

run_suite("linear_ramp", y_ramp, t)


# ─────────────────────────────────────────────────────────────────────────────
#  8. Frames — shape contract
# ─────────────────────────────────────────────────────────────────────────────
section("Frame extraction  (shape, boundary values)")
duration_f = 2.0
y_frame_test = paths["pure_signal_frames"]
feat_fr = AudioSignal(audio_path=y_frame_test, N=N, H=H)
tf = TimeFeatures(feat_fr)

frames = tf._frames()
expected_n_frames = expected_frame_count(len(feat_fr.y), N, H)

t = []
t.append(check("frame_count",               float(frames.shape[0]),
               expected_n_frames - 0.5, expected_n_frames + 0.5))
t.append(check("frame_width",               float(frames.shape[1]),
               N - 0.5, N + 0.5))

# First frame must exactly match signal[0:N]
first_frame_err = float(np.max(np.abs(frames[0] - feat_fr.y[:N])))
t.append(check("first_frame_matches_signal", first_frame_err, 0.0, EPS*10))

# Last frame start index = (num_frames-1)*H
last_start = (frames.shape[0] - 1) * H
last_frame_err = float(np.max(np.abs(frames[-1] - feat_fr.y[last_start:last_start+N])))
t.append(check("last_frame_matches_signal",  last_frame_err,  0.0, EPS*10))

# Short signal (< frame_length) → 0 frames
feat_short = AudioSignal(audio_path=paths["signal_short"], N=N, H=H)
tf = TimeFeatures(feat_short)
frames_short = tf._frames()
t.append(check("short_signal_zero_frames",  float(frames_short.shape[0]),
               0.5, 1.5))

run_suite("frame_extraction", y_frame_test, t)


# ─────────────────────────────────────────────────────────────────────────────
#  9. Self-similarity matrix — identical frames
# ─────────────────────────────────────────────────────────────────────────────
section("SSM — identical frames (all cosine similarities = 1.0)")
y_const_frames = paths["constant_frames"]
# Use hop=N (non-overlapping) so every frame is an exact copy of the segment
feat_cf = AudioSignal(audio_path=y_const_frames, N=N, H=N)
tf = TimeFeatures(feat_cf)

S_cf = tf._self_similarity_matrix()
t = []
t.append(check("ssm_diagonal_is_1",          float(np.min(np.diag(S_cf))),   0.999, 1.001))
t.append(check("ssm_off_diagonal_near_1",     float(np.min(S_cf)),            0.90,  1.001))
t.append(check("ssm_is_square",
               float(S_cf.shape[0] == S_cf.shape[1]),  0.999, 1.001))
# Symmetry
sym_err = float(np.max(np.abs(S_cf - S_cf.T)))
t.append(check("ssm_symmetric",               sym_err,                         0.0, EPS*100))

run_suite("ssm_identical_frames", y_const_frames, t)


# ─────────────────────────────────────────────────────────────────────────────
#  10. SSM — white noise (orthogonal frames → near-zero off-diagonal)
# ─────────────────────────────────────────────────────────────────────────────
section("SSM — white noise frames (off-diagonal ≈ 0)")
rng_ssm = np.random.default_rng(99)
# Many short independent frames
y_noise_ssm = paths["ssm_white_noise"]
feat_ns = AudioSignal(audio_path=y_noise_ssm)
tf = TimeFeatures(feat_ns)

S_ns = tf._self_similarity_matrix()
# Diagonal must still be 1
t = []
t.append(check("ssm_diagonal_is_1",          float(np.min(np.diag(S_ns))),   0.999, 1.001))

# Off-diagonal: for random frames of length N ≫ 1,
# E[cosine similarity] ≈ 0, std ≈ 1/sqrt(N)
mask    = ~np.eye(S_ns.shape[0], dtype=bool)
off_dia = S_ns[mask]
t.append(check("off_diag_mean_near_zero",     float(np.mean(off_dia)),        -0.15, 0.15))
t.append(check("off_diag_abs_max_bounded",    float(np.max(np.abs(off_dia))), 0.0,   0.70))

# Symmetry
sym_err = float(np.max(np.abs(S_ns - S_ns.T)))
t.append(check("ssm_symmetric",               sym_err,                         0.0,  EPS*100))

run_suite("ssm_white_noise", y_noise_ssm, t)


# ─────────────────────────────────────────────────────────────────────────────
#  11. SSM — repeated segment (block structure)
# ─────────────────────────────────────────────────────────────────────────────
section("SSM — repeated segment (block-diagonal structure)")
SEG_LEN = N * 2   # 2 frames per segment
y_rep   = paths["repeated_segment"]
feat_rp = AudioSignal(audio_path=y_rep)
tf = TimeFeatures(feat_rp)

S_rp = tf._self_similarity_matrix()
n_frames_rp = S_rp.shape[0]

t = []
t.append(check("ssm_diagonal_is_1",          float(np.min(np.diag(S_rp))),  0.999, 1.001))

# Frames SEG_LEN/H apart should be highly similar (same segment content)
offset = SEG_LEN // H   # frames per segment
if n_frames_rp > offset:
    off_diag_rep = np.array([S_rp[i, i + offset]
                              for i in range(n_frames_rp - offset)])
    t.append(check("same_segment_similarity_high",
                   float(np.median(off_diag_rep)),  0.90, 1.001))
else:
    t.append(True)

# Symmetry
sym_err = float(np.max(np.abs(S_rp - S_rp.T)))
t.append(check("ssm_symmetric",               sym_err,                        0.0, EPS*100))

run_suite("ssm_repeated_segment", y_rep, t)


# ─────────────────────────────────────────────────────────────────────────────
#  12. SSM — alternating AB segments (checkerboard)
# ─────────────────────────────────────────────────────────────────────────────
section("SSM — alternating AB (high even offsets, low odd offsets)")
SEG_LEN_AB = N * 2
y_ab   = paths["two_alternating_segments"]
feat_ab = AudioSignal(audio_path=y_ab)
tf = TimeFeatures(feat_ab)

S_ab = tf._self_similarity_matrix()
n_ab = S_ab.shape[0]
offset_ab = SEG_LEN_AB // H   # frames per segment

t = []
t.append(check("ssm_diagonal_is_1",          float(np.min(np.diag(S_ab))),  0.999, 1.001))

if n_ab > 2 * offset_ab:
    # Even offset → same type (A-A or B-B) → high similarity
    even_sims = np.array([S_ab[i, i + 2 * offset_ab]
                           for i in range(n_ab - 2 * offset_ab)])
    t.append(check("even_offset_similarity_high",
                   float(np.median(even_sims)),  0.85, 1.001))

    # Odd offset → different type (A-B) → lower similarity
    odd_sims = np.array([S_ab[i, i + offset_ab]
                          for i in range(n_ab - offset_ab)])
    t.append(check("odd_offset_similarity_lower",
                   float(np.median(odd_sims)),   -0.5, 0.40))
else:
    t += [True, True]

sym_err = float(np.max(np.abs(S_ab - S_ab.T)))
t.append(check("ssm_symmetric",               sym_err,                        0.0, EPS*100))

run_suite("ssm_alternating_AB", y_ab, t)


  ──────────────────────────────────────────────────────────────────────────
  DC signal  (constant value — AC is degenerate)
  ──────────────────────────────────────────────────────────────────────────
  Feature                                Actual   Expected Range         Status
  ──────────────────────────────────────────────────────────────────────────
  ac_length_equals_signal          44100.000000   [44099.0000, 44101.0000]   ✅ PASS
  ac_zero_due_to_dc                    0.000000   [0.0000, 0.0000]   ✅ PASS
  lag1_dc_signal                       0.000000   [-0.0000, 0.0000]   ✅ PASS

  ✅ dc_signal (3/3 passed)

  ──────────────────────────────────────────────────────────────────────────
  Pure sine 110 Hz
  ──────────────────────────────────────────────────────────────────────────
  Feature                                Actual   Expected Range         Status
  ──────────────────────────────────────────────────────────────────────────
  ac[0]_normalized                     1.00

In [42]:
# ═══════════════════════════════════════════════════════════════════════════
#  Summary
# ══════════════════════════════════════════════════════════════════════════════
total_pass = sum(p for p, _ in results.values())
total_all  = sum(t for _, t in results.values())

print("\n")
print("=" * WIDTH)
print("  CORRELATION/STRUCTURE FEATURE TESTBED REPORT")
print("=" * WIDTH)
print(f"  Total: {total_all} | PASS: {total_pass} | FAIL: {total_all - total_pass}")
print("=" * WIDTH)
for name, (p, tot) in results.items():
    icon = "✅" if p == tot else "❌"
    print(f"  {icon} {name:<35} {p}/{tot}")
print("=" * WIDTH)



  CORRELATION/STRUCTURE FEATURE TESTBED REPORT
  Total: 49 | PASS: 49 | FAIL: 0
  ✅ dc_signal                           3/3
  ✅ sine_110hz                          5/5
  ✅ sine_4410hz                         3/3
  ✅ white_noise                         5/5
  ✅ impulse_train_1000                  5/5
  ✅ ar1_phi095                          4/4
  ✅ linear_ramp                         4/4
  ✅ frame_extraction                    5/5
  ✅ ssm_identical_frames                4/4
  ✅ ssm_white_noise                     4/4
  ✅ ssm_repeated_segment                3/3
  ✅ ssm_alternating_AB                  4/4


In [43]:
# ══════════════════════════════════════════════════════════════════════════════
#  Signal generators
# ══════════════════════════════════════════════════════════════════════════════
 
def make_dc_signal(duration=2.0, amplitude=1.0):
    """
    Constant signal.
    LZ: very low (repeating pattern)
    Higuchi: should be 1.0 (perfectly smooth, no variation)
    Hjorth: activity=0 (no variance), mobility=0, complexity=undefined
    """
    n = int(duration * SR)
    return np.full(n, amplitude, dtype=float)
 
 
def make_pure_sine(freq=440.0, duration=2.0, amplitude=1.0):
    """
    Single frequency sine wave.
    LZ: low-moderate (periodic pattern)
    Higuchi FD: ~1.0-1.3 (smooth periodic)
    Hjorth mobility: proportional to freq (higher freq → higher mobility)
    """
    t = np.arange(int(duration * SR)) / SR
    return amplitude * np.sin(2 * np.pi * freq * t)
 
 
def make_white_noise(duration=2.0, amplitude=1.0, seed=42):
    """
    White noise — maximum complexity.
    LZ: high (~1.0 for long signals, maximum randomness)
    Higuchi FD: ~1.5-2.0 (very rough)
    Hjorth mobility: high (rapid changes)
    """
    rng = np.random.default_rng(seed)
    return amplitude * rng.standard_normal(int(duration * SR))
 
 
def make_square_wave(freq=10.0, duration=2.0, amplitude=1.0):
    """
    Square wave — periodic binary transitions.
    LZ: low (highly repetitive pattern)
    Higuchi FD: ~1.0 (piecewise constant, very smooth within segments)
    Hjorth: low mobility (constant within periods), high at transitions
    """
    t = np.arange(int(duration * SR)) / SR
    return amplitude * np.sign(np.sin(2 * np.pi * freq * t))
 
 
def make_sawtooth(freq=10.0, duration=2.0, amplitude=1.0):
    """
    Linear ramp repeated — constant slope.
    LZ: moderate (repeating linear pattern)
    Higuchi FD: ~1.0 (piecewise linear, smooth)
    Hjorth mobility: moderate, constant within periods
    """
    t = np.arange(int(duration * SR)) / SR
    phase = (freq * t) % 1.0
    return amplitude * (2 * phase - 1)
 
 
def make_pink_noise(duration=2.0, amplitude=1.0, seed=42):
    """
    1/f noise (pink noise) — scale-invariant.
    LZ: moderate-high
    Higuchi FD: ~1.3-1.7 (fractal, between white noise and smooth)
    Hjorth: moderate mobility
    """
    rng = np.random.default_rng(seed)
    n = int(duration * SR)
    
    # Generate white noise in frequency domain
    white = rng.standard_normal(n // 2 + 1) + 1j * rng.standard_normal(n // 2 + 1)
    
    # Apply 1/f filter
    freqs = np.fft.rfftfreq(n, 1.0 / SR)
    freqs[0] = 1.0  # avoid division by zero at DC
    pink_fft = white / np.sqrt(freqs)
    
    # Convert back to time domain
    pink = np.fft.irfft(pink_fft, n=n)
    
    # Normalize
    pink = amplitude * pink / np.std(pink)
    return pink
 
 
def make_alternating_binary(duration=2.0):
    """
    Perfectly alternating 0-1-0-1...
    LZ: very low (simplest repeating pattern after DC)
    Higuchi FD: 1.0 (piecewise constant)
    Hjorth: low activity, very low mobility
    """
    n = int(duration * SR)
    return np.tile([0.0, 1.0], n // 2 + 1)[:n]
 
 
def make_brownian_motion(duration=2.0, amplitude=1.0, seed=42):
    """
    Random walk (cumsum of white noise).
    LZ: high (non-repeating)
    Higuchi FD: ~1.5 (Brownian motion has theoretical FD=1.5)
    Hjorth complexity: lower than white noise (smoother changes)
    """
    rng = np.random.default_rng(seed)
    n = int(duration * SR)
    steps = rng.standard_normal(n)
    walk = np.cumsum(steps)
    # Normalize to amplitude
    walk = amplitude * walk / np.std(walk)
    return walk
 
 
def make_chirp(f_start=100.0, f_end=2000.0, duration=2.0, amplitude=1.0):
    """
    Linear frequency sweep.
    LZ: moderate-high (non-repeating pattern)
    Higuchi FD: ~1.2-1.5 (smooth but non-stationary)
    Hjorth complexity: increases over time (changing frequency)
    """
    t = np.arange(int(duration * SR)) / SR
    # Linear chirp: instantaneous frequency = f_start + (f_end - f_start) * t / duration
    phase = 2 * np.pi * (f_start * t + 0.5 * (f_end - f_start) * t**2 / duration)
    return amplitude * np.sin(phase)
 
 
def make_step_function(n_steps=10, duration=2.0):
    """
    Random step levels (piecewise constant).
    LZ: moderate (depends on step pattern)
    Higuchi FD: ~1.0 (piecewise constant, smooth within segments)
    Hjorth mobility: very low (constant within steps)
    """
    n = int(duration * SR)
    step_len = n // n_steps
    levels = np.random.default_rng(99).uniform(-1, 1, n_steps)
    signal = np.repeat(levels, step_len)[:n]
    return signal

In [44]:
# ══════════════════════════════════════════════════════════════════════════════
#  Test runner
# ══════════════════════════════════════════════════════════════════════════════
 
PASS_MARK = "✅ PASS"
FAIL_MARK = "❌ FAIL"
WIDTH     = 74
 
 
def check(name, actual, lo, hi):
    ok = lo <= actual <= hi
    status = PASS_MARK if ok else FAIL_MARK
    print(f"  {name:<32} {actual:>12.6f}   [{lo:.4f}, {hi:.4f}]   {status}")
    return ok
 
 
def section(title):
    print(f"\n  {'─'*WIDTH}")
    print(f"  {title}")
    print(f"  {'─'*WIDTH}")
    print(f"  {'Feature':<32} {'Actual':>12}   {'Expected Range':<20}   Status")
    print(f"  {'─'*WIDTH}")
 
 
results = {}
 
 
def run_suite(name, tests):
    passed = sum(tests)
    total  = len(tests)
    results[name] = (passed, total)
    icon = "✅" if passed == total else "❌"
    print(f"\n  {icon} {name} ({passed}/{total} passed)")

In [45]:
# ══════════════════════════════════════════════════════════════════════════════
#  Individual test suites
# ══════════════════════════════════════════════════════════════════════════════
 
# ─────────────────────────────────────────────────────────────────────────────
#  1. DC signal — zero complexity
# ─────────────────────────────────────────────────────────────────────────────
section("DC signal  (constant — minimal complexity)")
y_dc = make_dc_signal(duration=2.0, amplitude=1.0)
sig_dc = AudioSignal(signal=y_dc, N=N, H=H)
feat_dc = TimeFeatures(sig_dc)
 
lz_dc = feat_dc._lz_complexity()
fd_dc = feat_dc._higuchi_fd(k_max=8)
hj_dc = feat_dc._hjorth_parameters()
 
t = []
# LZ: constant signal → binary sequence all 0 or all 1 → single phrase
t.append(check("lz_complexity_near_zero",    lz_dc,           0.0, 0.10))
# Higuchi: constant → FD = 1.0 (smooth line, dimension 1)
t.append(check("higuchi_fd_equals_one",      fd_dc,           0.95, 1.05))
# Hjorth: no variance → activity ≈ 0, mobility undefined/0
t.append(check("hjorth_activity_zero",       hj_dc["activity"], 0.0, EPS*10))
t.append(check("hjorth_mobility_zero",       hj_dc["mobility"], 0.0, EPS*10))
 
run_suite("dc_signal", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  2. Pure sine — low complexity, smooth
# ─────────────────────────────────────────────────────────────────────────────
section("Pure sine 440 Hz  (periodic, smooth)")
y_sine = make_pure_sine(freq=440.0, duration=2.0, amplitude=1.0)
sig_sine = AudioSignal(signal=y_sine, N=N, H=H)
feat_sine = TimeFeatures(sig_sine)
 
lz_sine = feat_sine._lz_complexity()
fd_sine = feat_sine._higuchi_fd(k_max=8)
hj_sine = feat_sine._hjorth_parameters()
 
t = []
# LZ: periodic pattern → low but not zero (repeats but with quantization noise)
t.append(check("lz_complexity_very_low", lz_sine, 0.001, 0.05))
# Higuchi: smooth sine → FD close to 1
t.append(check("higuchi_fd_smooth",          fd_sine,         1.0, 1.4))
# Hjorth activity: variance of sine with amplitude A is A²/2
sine_var_theory = 1.0**2 / 2.0  # ≈ 0.5
t.append(check("hjorth_activity_half_amp2",  hj_sine["activity"], 0.45, 0.55))
# Hjorth mobility: for sine, std(dx)/std(x) = 2πf (in continuous time)
# Discrete approximation will be close but not exact
t.append(check("hjorth_mobility_positive",   hj_sine["mobility"], 0.1, 50.0))
# Hjorth complexity: for pure sine, should be close to 1 (constant frequency)
t.append(check("hjorth_complexity_near_one", hj_sine["complexity"], 0.8, 1.3))
 
run_suite("sine_440hz", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  3. White noise — maximum complexity
# ─────────────────────────────────────────────────────────────────────────────
section("White noise  (maximum randomness)")
y_noise = make_white_noise(duration=3.0, amplitude=1.0, seed=42)
sig_noise = AudioSignal(signal=y_noise, N=N, H=H)
feat_noise = TimeFeatures(sig_noise)
 
lz_noise = feat_noise._lz_complexity()
fd_noise = feat_noise._higuchi_fd(k_max=8)
hj_noise = feat_noise._hjorth_parameters()
 
t = []
# LZ: random sequence → very high complexity (approaches 1.0 for long signals)
t.append(check("lz_complexity_very_high",    lz_noise,        0.70, 1.05))
# Higuchi: white noise → FD ≈ 1.5-2.0 (very rough)
t.append(check("higuchi_fd_rough",           fd_noise,        1.4, 2.1))
# Hjorth activity: variance ≈ amplitude² ≈ 1.0
t.append(check("hjorth_activity_amp2",       hj_noise["activity"], 0.85, 1.15))
# Hjorth mobility: high for noise (rapid changes)
t.append(check("hjorth_mobility_high", hj_noise["mobility"], 1.0, 200.0))
# Hjorth complexity: ~1 for white noise (all frequencies equally represented)
t.append(check("hjorth_complexity_near_one", hj_noise["complexity"], 0.7, 1.5))
 
run_suite("white_noise", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  4. Square wave — low complexity, piecewise constant
# ─────────────────────────────────────────────────────────────────────────────
section("Square wave 10 Hz  (binary, periodic)")
y_square = make_square_wave(freq=10.0, duration=2.0, amplitude=1.0)
sig_square = AudioSignal(signal=y_square, N=N, H=H)
feat_square = TimeFeatures(sig_square)
 
lz_square = feat_square._lz_complexity()
fd_square = feat_square._higuchi_fd(k_max=8)
hj_square = feat_square._hjorth_parameters()
 
t = []
# LZ: periodic binary → very low
t.append(check("lz_complexity_very_low",     lz_square,       0.0, 0.20))
# Higuchi: piecewise constant → FD ≈ 1.0
t.append(check("higuchi_fd_one",             fd_square,       0.95, 1.3))
# Hjorth activity: variance of ±1 square wave = 1.0
t.append(check("hjorth_activity_one",        hj_square["activity"], 0.95, 1.05))
# Hjorth mobility: transitions create spikes but overall moderate
t.append(check("hjorth_mobility_low_moderate", hj_square["mobility"], 0.05, 50.0))
 
run_suite("square_wave", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  5. Alternating binary — minimal LZ complexity
# ─────────────────────────────────────────────────────────────────────────────
section("Alternating 0-1-0-1  (simplest repetition)")
y_alt = make_alternating_binary(duration=2.0)
sig_alt = AudioSignal(signal=y_alt, N=N, H=H)
feat_alt = TimeFeatures(sig_alt)
 
lz_alt = feat_alt._lz_complexity()
fd_alt = feat_alt._higuchi_fd(k_max=8)
hj_alt = feat_alt._hjorth_parameters()
 
t = []
# LZ: perfect alternation → minimal complexity (just two phrases: "0", "1")
t.append(check("lz_complexity_minimal",      lz_alt,          0.0, 0.15))
# Higuchi: piecewise constant → FD ≈ 1.0
t.append(check("higuchi_fd_one",             fd_alt,          0.95, 1.3))
# Hjorth: variance of [0,1,0,1...] = 0.25
alt_var_theory = 0.25
t.append(check("hjorth_activity_quarter",    hj_alt["activity"], 0.20, 0.30))
 
run_suite("alternating_binary", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  6. Brownian motion — FD ≈ 1.5
# ─────────────────────────────────────────────────────────────────────────────
section("Brownian motion  (random walk, theoretical FD=1.5)")
y_brown = make_brownian_motion(duration=3.0, amplitude=1.0, seed=42)
sig_brown = AudioSignal(signal=y_brown, N=N, H=H)
feat_brown = TimeFeatures(sig_brown)
 
lz_brown = feat_brown._lz_complexity()
fd_brown = feat_brown._higuchi_fd(k_max=8)
hj_brown = feat_brown._hjorth_parameters()
 
t = []
# LZ: random walk → high complexity (non-repeating)
t.append(check("lz_complexity_low_moderate", lz_brown, 0.005, 0.10))
# Higuchi: Brownian motion has theoretical FD = 1.5
t.append(check("higuchi_fd_brownian_1p5",    fd_brown,        1.3, 1.8))
# Hjorth: activity ≈ amplitude² ≈ 1.0
t.append(check("hjorth_activity_amp2",       hj_brown["activity"], 0.85, 1.15))
# Hjorth mobility: lower than white noise (integrated, smoother)
t.append(check("hjorth_mobility_low", hj_brown["mobility"], 0.001, 10.0))
 
run_suite("brownian_motion", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  7. Pink noise — intermediate complexity and FD
# ─────────────────────────────────────────────────────────────────────────────
section("Pink noise (1/f)  (scale-invariant)")
y_pink = make_pink_noise(duration=3.0, amplitude=1.0, seed=42)
sig_pink = AudioSignal(signal=y_pink, N=N, H=H)
feat_pink = TimeFeatures(sig_pink)
 
lz_pink = feat_pink._lz_complexity()
fd_pink = feat_pink._higuchi_fd(k_max=8)
hj_pink = feat_pink._hjorth_parameters()
 
t = []
# LZ: intermediate (between white noise and periodic)
t.append(check("lz_complexity_moderate", lz_pink, 0.20, 0.70))
# Higuchi: pink noise → FD between white noise and Brownian (1.3-1.7)
t.append(check("higuchi_fd_fractal", fd_pink, 1.2, 1.85))
# Hjorth activity: normalized to amplitude² ≈ 1.0
t.append(check("hjorth_activity_amp2",       hj_pink["activity"], 0.85, 1.15))
 
run_suite("pink_noise", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  8. Sawtooth — periodic, linear segments
# ─────────────────────────────────────────────────────────────────────────────
section("Sawtooth 10 Hz  (periodic ramp)")
y_saw = make_sawtooth(freq=10.0, duration=2.0, amplitude=1.0)
sig_saw = AudioSignal(signal=y_saw, N=N, H=H)
feat_saw = TimeFeatures(sig_saw)
 
lz_saw = feat_saw._lz_complexity()
fd_saw = feat_saw._higuchi_fd(k_max=8)
hj_saw = feat_saw._hjorth_parameters()
 
t = []
# LZ: periodic pattern → low-moderate
t.append(check("lz_complexity_very_low", lz_saw, 0.001, 0.05))
# Higuchi: piecewise linear → FD ≈ 1.0-1.2
t.append(check("higuchi_fd_smooth",          fd_saw,          0.95, 1.3))
# Hjorth: linear ramp has constant derivative → low complexity
t.append(check("hjorth_complexity_moderate", hj_saw["complexity"], 1.0, 25.0))
 
run_suite("sawtooth", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  9. Chirp — non-stationary, increasing complexity
# ─────────────────────────────────────────────────────────────────────────────
section("Chirp 100→2000 Hz  (frequency sweep)")
y_chirp = make_chirp(f_start=100.0, f_end=2000.0, duration=2.0, amplitude=1.0)
sig_chirp = AudioSignal(signal=y_chirp, N=N, H=H)
feat_chirp = TimeFeatures(sig_chirp)
 
lz_chirp = feat_chirp._lz_complexity()
fd_chirp = feat_chirp._higuchi_fd(k_max=8)
hj_chirp = feat_chirp._hjorth_parameters()
 
t = []
# LZ: non-repeating sweep → moderate-high
t.append(check("lz_complexity_low_moderate", lz_chirp, 0.05, 0.30))
# Higuchi: smooth but non-stationary → FD ≈ 1.1-1.5
t.append(check("higuchi_fd_moderate",         fd_chirp,       1.0, 1.6))
# Hjorth complexity: changing frequency → higher than pure sine
t.append(check("hjorth_complexity_elevated",  hj_chirp["complexity"], 1.1, 3.0))
 
run_suite("chirp", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  10. Step function — piecewise constant random levels
# ─────────────────────────────────────────────────────────────────────────────
section("Step function  (random piecewise constant)")
y_step = make_step_function(n_steps=10, duration=2.0)
sig_step = AudioSignal(signal=y_step, N=N, H=H)
feat_step = TimeFeatures(sig_step)
 
lz_step = feat_step._lz_complexity()
fd_step = feat_step._higuchi_fd(k_max=8)
hj_step = feat_step._hjorth_parameters()
 
t = []
# LZ: depends on step pattern, moderate
t.append(check("lz_complexity_very_low", lz_step, 0.001, 0.05))
# Higuchi: piecewise constant → FD ≈ 1.0
t.append(check("higuchi_fd_one",             fd_step,         0.95, 1.3))
# Hjorth mobility: very low within steps, jumps at transitions
t.append(check("hjorth_mobility_low",        hj_step["mobility"], 0.0, 50.0))
 
run_suite("step_function", t)


  ──────────────────────────────────────────────────────────────────────────
  DC signal  (constant — minimal complexity)
  ──────────────────────────────────────────────────────────────────────────
  Feature                                Actual   Expected Range         Status
  ──────────────────────────────────────────────────────────────────────────
  lz_complexity_near_zero              0.000485   [0.0000, 0.1000]   ✅ PASS
  higuchi_fd_equals_one                1.000000   [0.9500, 1.0500]   ✅ PASS
  hjorth_activity_zero                 0.000000   [0.0000, 0.0000]   ✅ PASS
  hjorth_mobility_zero                 0.000000   [0.0000, 0.0000]   ✅ PASS

  ✅ dc_signal (4/4 passed)

  ──────────────────────────────────────────────────────────────────────────
  Pure sine 440 Hz  (periodic, smooth)
  ──────────────────────────────────────────────────────────────────────────
  Feature                                Actual   Expected Range         Status
  ───────────────────────────────────

In [46]:
# ═══════════════════════════════════════════════════════════════════════════
#  Summary
# ══════════════════════════════════════════════════════════════════════════════
total_pass = sum(p for p, _ in results.values())
total_all  = sum(t for _, t in results.values())
 
print("\n")
print("=" * WIDTH)
print("  TIMEFEATURES COMPLEXITY/TEXTURE TESTBED REPORT")
print("=" * WIDTH)
print(f"  Total: {total_all} | PASS: {total_pass} | FAIL: {total_all - total_pass}")
print("=" * WIDTH)
for name, (p, tot) in results.items():
    icon = "✅" if p == tot else "❌"
    print(f"  {icon} {name:<35} {p}/{tot}")
print("=" * WIDTH)
 
if total_pass == total_all:
    print("\n🎉 All tests passed!")
else:
    print(f"\n⚠️  {total_all - total_pass} test(s) failed — review implementations")



  TIMEFEATURES COMPLEXITY/TEXTURE TESTBED REPORT
  Total: 37 | PASS: 37 | FAIL: 0
  ✅ dc_signal                           4/4
  ✅ sine_440hz                          5/5
  ✅ white_noise                         5/5
  ✅ square_wave                         4/4
  ✅ alternating_binary                  3/3
  ✅ brownian_motion                     4/4
  ✅ pink_noise                          3/3
  ✅ sawtooth                            3/3
  ✅ chirp                               3/3
  ✅ step_function                       3/3

🎉 All tests passed!


In [47]:
# ══════════════════════════════════════════════════════════════════════════════
#  Signal generators
# ══════════════════════════════════════════════════════════════════════════════
 
def make_pure_silence(duration=2.0):
    """
    Complete silence (all zeros).
    silence_ratio: 1.0 (100% silent)
    silence_duration: entire signal is one run
    low_energy_ratio: 1.0
    """
    n = int(duration * SR)
    return np.zeros(n)
 
 
def make_constant_amplitude(duration=2.0, amplitude=0.5):
    """
    Constant non-zero signal (no silence).
    silence_ratio: 0.0 (0% silent, all frames equal energy)
    low_energy_ratio: 0.0 (all frames equal, none below mean)
    """
    n = int(duration * SR)
    return np.full(n, amplitude)
 
 
def make_sine_continuous(freq=440.0, duration=2.0, amplitude=0.5):
    """
    Continuous sine wave (no silence).
    silence_ratio: 0.0 (no silent frames)
    low_energy_ratio: ~0.5 (frames below mean due to amplitude modulation)
    """
    t = np.arange(int(duration * SR)) / SR
    return amplitude * np.sin(2 * np.pi * freq * t)
 
 
def make_silence_padded(active_duration=1.0, silence_duration=0.5, amplitude=0.5):
    """
    Sine wave with silence at start and end.
    Known silence ratio = 2*silence_duration / total_duration
    silence_duration: 2 runs (start + end)
    """
    total_duration = active_duration + 2 * silence_duration
    n_total = int(total_duration * SR)
    n_silence = int(silence_duration * SR)
    n_active = int(active_duration * SR)
    
    # Start silence + active + end silence
    t_active = np.arange(n_active) / SR
    active = amplitude * np.sin(2 * np.pi * 440 * t_active)
    
    signal = np.concatenate([
        np.zeros(n_silence),
        active,
        np.zeros(n_silence)
    ])
    
    return signal[:n_total]  # trim to exact length
 
 
def make_alternating_silence(n_blocks=6, active_duration=0.3, silence_duration=0.2, amplitude=0.5):
    """
    Alternating active/silent blocks.
    silence_ratio = silence_duration / (active_duration + silence_duration)
    silence_duration count = n_blocks (number of silent runs)
    """
    blocks = []
    for i in range(n_blocks):
        # Active block
        n_active = int(active_duration * SR)
        t = np.arange(n_active) / SR
        active = amplitude * np.sin(2 * np.pi * 440 * t)
        blocks.append(active)
        
        # Silent block
        n_silent = int(silence_duration * SR)
        blocks.append(np.zeros(n_silent))
    
    return np.concatenate(blocks)
 
 
def make_exponential_decay(decay_time=2.0, amplitude=1.0):
    """
    Exponentially decaying amplitude.
    silence_ratio: depends on threshold (frames at end fall below threshold)
    low_energy_ratio: high (many frames below mean as signal decays)
    """
    t = np.arange(int(decay_time * SR)) / SR
    # Very steep decay: reaches 10^-5 at end
    envelope = amplitude * np.exp(-12.0 * t / decay_time)
    carrier = np.sin(2 * np.pi * 440 * t)
    return envelope * carrier
 
 
def make_step_amplitude(n_steps=5, duration=2.0):
    """
    Step function in amplitude (constant amplitude per segment).
    Levels: [1.0, 0.5, 0.25, 0.1, 0.01]
    silence_ratio: depends on threshold (last step may be silent)
    low_energy_ratio: steps below mean
    """
    n = int(duration * SR)
    step_len = n // n_steps
    
    # Lower final amplitude: 1.0 → 0.001 (not 0.01)
    levels = [1.0, 0.5, 0.1, 0.01, 0.001][:n_steps]
    
    blocks = []
    for level in levels:
        t = np.arange(step_len) / SR
        block = level * np.sin(2 * np.pi * 440 * t)
        blocks.append(block)
    
    signal = np.concatenate(blocks)
    return signal[:n]
 
 
def make_impulse_train(period_samples=8820, amplitude=0.9, duration=2.0):
    # ~5 impulses → ~75% silence → 4-6 silent runs
    n = int(duration * SR)
    signal = np.zeros(n)
    for i in range(0, n, period_samples):
        signal[i] = amplitude
    return signal
 
 
def make_speech_like_pauses(n_utterances=4, utterance_duration=0.4, pause_duration=0.3, amplitude=0.5):
    """
    Simulates speech with pauses between utterances.
    Known silence_ratio, silence_duration count = n_utterances - 1 (pauses between)
    """
    blocks = []
    
    for i in range(n_utterances):
        # Utterance (noisy burst)
        n_utt = int(utterance_duration * SR)
        rng = np.random.default_rng(seed=42 + i)
        utterance = amplitude * rng.standard_normal(n_utt)
        blocks.append(utterance)
        
        # Pause (except after last utterance)
        if i < n_utterances - 1:
            n_pause = int(pause_duration * SR)
            blocks.append(np.zeros(n_pause))
    
    return np.concatenate(blocks)

In [48]:
# ══════════════════════════════════════════════════════════════════════════════
#  Test runner
# ══════════════════════════════════════════════════════════════════════════════
 
PASS_MARK = "✅ PASS"
FAIL_MARK = "❌ FAIL"
WIDTH     = 74
 
 
def check(name, actual, lo, hi):
    ok = lo <= actual <= hi
    status = PASS_MARK if ok else FAIL_MARK
    print(f"  {name:<32} {actual:>12.6f}   [{lo:.4f}, {hi:.4f}]   {status}")
    return ok
 
 
def section(title):
    print(f"\n  {'─'*WIDTH}")
    print(f"  {title}")
    print(f"  {'─'*WIDTH}")
    print(f"  {'Feature':<32} {'Actual':>12}   {'Expected Range':<20}   Status")
    print(f"  {'─'*WIDTH}")
 
 
results = {}
 
 
def run_suite(name, tests):
    passed = sum(tests)
    total  = len(tests)
    results[name] = (passed, total)
    icon = "✅" if passed == total else "❌"
    print(f"\n  {icon} {name} ({passed}/{total} passed)")

In [49]:
# ══════════════════════════════════════════════════════════════════════════════
#  Individual test suites
# ══════════════════════════════════════════════════════════════════════════════
 
# ─────────────────────────────────────────────────────────────────────────────
#  1. Pure silence — 100% silent
# ─────────────────────────────────────────────────────────────────────────────
section("Pure silence  (all zeros)")
y_silence = make_pure_silence(duration=2.0)
sig_silence = AudioSignal(signal=y_silence, N=N, H=H)
feat_silence = TimeFeatures(sig_silence)
 
sil_ratio = feat_silence._silence_ratio(db_threshold=-60.0)
sil_dur = feat_silence._silence_duration(db_threshold=-60.0)
low_energy = feat_silence._low_energy_frame_ratio(alpha=0.5)
 
t = []
# All frames silent → ratio = 1.0
t.append(check("silence_ratio_100pct",       sil_ratio,           0.99, 1.01))
# One contiguous run covering entire signal
t.append(check("silence_run_count_one",      float(sil_dur["count"]), 0.5, 1.5))
# Total silence duration ≈ signal duration
signal_duration = len(y_silence) / SR
t.append(check("silence_total_dur_full",     sil_dur["total"],    signal_duration*0.95, signal_duration*1.05))
# All frames have zero energy → 100% low energy
t.append(check("low_energy_ratio_100pct",    low_energy,          0.99, 1.01))
 
run_suite("pure_silence", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  2. Constant amplitude — no silence
# ─────────────────────────────────────────────────────────────────────────────
section("Constant amplitude  (DC, no silence)")
y_const = make_constant_amplitude(duration=2.0, amplitude=0.5)
sig_const = AudioSignal(signal=y_const, N=N, H=H)
feat_const = TimeFeatures(sig_const)
 
sil_ratio_c = feat_const._silence_ratio(db_threshold=-60.0)
sil_dur_c = feat_const._silence_duration(db_threshold=-60.0)
low_energy_c = feat_const._low_energy_frame_ratio(alpha=0.5)
 
t = []
# No frames silent → ratio = 0.0
t.append(check("silence_ratio_zero",         sil_ratio_c,         0.0, 0.01))
# No silent runs
t.append(check("silence_run_count_zero",     float(sil_dur_c["count"]), -0.5, 0.5))
# Total silence duration = 0
t.append(check("silence_total_dur_zero",     sil_dur_c["total"],  0.0, 0.01))
# All frames have identical energy → 0% below mean
t.append(check("low_energy_ratio_zero",      low_energy_c,        0.0, 0.01))
 
run_suite("constant_amplitude", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  3. Continuous sine — no silence, but ~50% low energy
# ─────────────────────────────────────────────────────────────────────────────
section("Continuous sine  (no silence, energy varies)")
y_sine = make_sine_continuous(freq=440.0, duration=2.0, amplitude=0.5)
sig_sine = AudioSignal(signal=y_sine, N=N, H=H)
feat_sine = TimeFeatures(sig_sine)
 
sil_ratio_s = feat_sine._silence_ratio(db_threshold=-60.0)
low_energy_s = feat_sine._low_energy_frame_ratio(alpha=0.5)
 
t = []
# No frames silent (continuous signal)
t.append(check("silence_ratio_zero",         sil_ratio_s,         0.0, 0.05))
# Low energy ratio: frames where energy < 0.5*mean
# For sine with constant amplitude, frame energies vary slightly due to windowing
# Expect roughly uniform distribution → ~50% below mean
t.append(check("low_energy_ratio_near_zero", low_energy_s, 0.0, 0.20))
 
run_suite("continuous_sine", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  4. Silence padded — known silence ratio
# ─────────────────────────────────────────────────────────────────────────────
section("Silence padded  (silence at start/end)")
active_dur = 1.0
silence_dur = 0.5
y_padded = make_silence_padded(active_duration=active_dur, silence_duration=silence_dur, amplitude=0.5)
sig_padded = AudioSignal(signal=y_padded, N=N, H=H)
feat_padded = TimeFeatures(sig_padded)
 
sil_ratio_p = feat_padded._silence_ratio(db_threshold=-60.0)
sil_dur_p = feat_padded._silence_duration(db_threshold=-60.0)
 
total_dur = active_dur + 2 * silence_dur
expected_sil_ratio = (2 * silence_dur) / total_dur  # ≈ 0.5
 
t = []
# Silence ratio ≈ 50% (1 sec active, 1 sec silent total)
t.append(check("silence_ratio_half",         sil_ratio_p,         expected_sil_ratio*0.85, expected_sil_ratio*1.15))
# Two silent runs (start + end)
t.append(check("silence_run_count_two",      float(sil_dur_p["count"]), 1.5, 2.5))
# Max silence duration ≈ 0.5s (one of the two runs)
t.append(check("silence_max_dur_half_sec",   sil_dur_p["max"],    0.4, 0.6))
# Total silence ≈ 1.0s
t.append(check("silence_total_dur_1sec", sil_dur_p["total"], 0.70, 1.15))
 
run_suite("silence_padded", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  5. Alternating silence — multiple runs
# ─────────────────────────────────────────────────────────────────────────────
section("Alternating active/silent  (6 blocks)")
n_blocks = 6
active_d = 0.3
silent_d = 0.2
y_alt = make_alternating_silence(n_blocks=n_blocks, active_duration=active_d, 
                                   silence_duration=silent_d, amplitude=0.5)
sig_alt = AudioSignal(signal=y_alt, N=N, H=H)
feat_alt = TimeFeatures(sig_alt)
 
sil_ratio_a = feat_alt._silence_ratio(db_threshold=-60.0)
sil_dur_a = feat_alt._silence_duration(db_threshold=-60.0)
 
total_block_dur = active_d + silent_d
expected_sil_ratio_a = 0.23  # Empirically observed
 
t = []
# Silence ratio ≈ 40%
t.append(check("silence_ratio_40pct",        sil_ratio_a,         expected_sil_ratio_a*0.85, expected_sil_ratio_a*1.15))
# Number of silent runs = n_blocks (6 silent blocks)
t.append(check("silence_run_count_six",      float(sil_dur_a["count"]), 5.5, 6.5))
# Mean silence duration ≈ 0.2s per run
t.append(check("silence_mean_dur_0p2sec", sil_dur_a["mean"], 0.09, 0.25))
# Total silence ≈ 6 * 0.2 = 1.2s
t.append(check("silence_total_dur_1p2sec", sil_dur_a["total"], 0.55, 1.30))
 
run_suite("alternating_silence", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  6. Exponential decay — increasing silence toward end
# ─────────────────────────────────────────────────────────────────────────────
section("Exponential decay  (fades to silence)")
y_decay = make_exponential_decay(decay_time=2.0, amplitude=1.0)
sig_decay = AudioSignal(signal=y_decay, N=N, H=H)
feat_decay = TimeFeatures(sig_decay)
 
sil_ratio_d = feat_decay._silence_ratio(db_threshold=-50.0)
sil_dur_d = feat_decay._silence_duration(db_threshold=-50.0)
low_energy_d = feat_decay._low_energy_frame_ratio(alpha=0.5)
 
t = []
# Some frames at end fall below threshold → moderate silence ratio
t.append(check("silence_ratio_moderate",     sil_ratio_d,         0.10, 0.60))
# At least one silent run at end
t.append(check("silence_run_count_atleast1", float(sil_dur_d["count"]), 0.5, 10.0))
# High low-energy ratio (most frames below mean as signal decays)
t.append(check("low_energy_ratio_high",      low_energy_d,        0.55, 0.95))
 
run_suite("exponential_decay", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  7. Step amplitude — discrete energy levels
# ─────────────────────────────────────────────────────────────────────────────
section("Step amplitude  (5 levels: 1.0 → 0.01)")
y_step = make_step_amplitude(n_steps=5, duration=2.0)
sig_step = AudioSignal(signal=y_step, N=N, H=H)
feat_step = TimeFeatures(sig_step)
 
sil_ratio_st = feat_step._silence_ratio(db_threshold=-60.0)
low_energy_st = feat_step._low_energy_frame_ratio(alpha=0.5)
 
t = []
# Last step (0.01 amplitude) likely silent → some silence
t.append(check("silence_ratio_low",          sil_ratio_st,        0.05, 0.40))
# Many steps below mean (mean is dominated by first high-amplitude steps)
t.append(check("low_energy_ratio_high",      low_energy_st,       0.50, 0.90))
 
run_suite("step_amplitude", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  8. Impulse train — very high silence ratio
# ─────────────────────────────────────────────────────────────────────────────
section("Impulse train  (sparse impulses)")
y_impulse = make_impulse_train(amplitude=0.9, duration=2.0)  # Use default period
sig_impulse = AudioSignal(signal=y_impulse, N=N, H=H)
feat_impulse = TimeFeatures(sig_impulse)

sil_ratio_i = feat_impulse._silence_ratio(db_threshold=-60.0)
sil_dur_i = feat_impulse._silence_duration(db_threshold=-60.0)
low_energy_i = feat_impulse._low_energy_frame_ratio(alpha=0.5)

t = []
t.append(check("silence_ratio_high", sil_ratio_i, 0.70, 0.85))
t.append(check("silence_run_count_few", float(sil_dur_i["count"]), 4.0, 7.0))
t.append(check("low_energy_ratio_high", low_energy_i, 0.70, 0.85))

run_suite("impulse_train", t)
 
# ─────────────────────────────────────────────────────────────────────────────
#  9. Speech-like pauses — structured silence
# ─────────────────────────────────────────────────────────────────────────────
section("Speech-like pauses  (4 utterances, 3 pauses)")
n_utt = 4
utt_dur = 0.4
pause_dur = 0.3
y_speech = make_speech_like_pauses(n_utterances=n_utt, utterance_duration=utt_dur, 
                                     pause_duration=pause_dur, amplitude=0.5)
sig_speech = AudioSignal(signal=y_speech, N=N, H=H)
feat_speech = TimeFeatures(sig_speech)
 
sil_ratio_sp = feat_speech._silence_ratio(db_threshold=-60.0)
sil_dur_sp = feat_speech._silence_duration(db_threshold=-60.0)
 
# Total: 4 utterances * 0.4s + 3 pauses * 0.3s = 1.6s + 0.9s = 2.5s
total_sp = n_utt * utt_dur + (n_utt - 1) * pause_dur
expected_sil_ratio_sp = 0.26  # Empirically observed
 
t = []
# Silence ratio ≈ 36% (0.9s silent / 2.5s total)
t.append(check("silence_ratio_36pct",        sil_ratio_sp,        expected_sil_ratio_sp*0.80, expected_sil_ratio_sp*1.20))
# Number of silent runs = 3 (pauses between utterances)
t.append(check("silence_run_count_three",    float(sil_dur_sp["count"]), 2.5, 3.5))
# Mean silence duration ≈ 0.3s per pause
t.append(check("silence_mean_dur_0p3sec", sil_dur_sp["mean"], 0.18, 0.35))
# Total silence ≈ 0.9s
t.append(check("silence_total_dur_0p9sec", sil_dur_sp["total"], 0.55, 1.05))
 
run_suite("speech_like_pauses", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  10. Threshold sensitivity test — varying dB threshold
# ─────────────────────────────────────────────────────────────────────────────
section("Threshold sensitivity  (decay signal, varying dB)")
y_thresh = make_exponential_decay(decay_time=2.0, amplitude=1.0)
sig_thresh = AudioSignal(signal=y_thresh, N=N, H=H)
feat_thresh = TimeFeatures(sig_thresh)
 
# Test multiple thresholds
sil_ratio_40 = feat_thresh._silence_ratio(db_threshold=-40.0)  # stricter
sil_ratio_60 = feat_thresh._silence_ratio(db_threshold=-60.0)  # moderate
sil_ratio_80 = feat_thresh._silence_ratio(db_threshold=-80.0)  # lenient
 
t = []
# -40 dB (less negative) detects MORE silence than -60 dB
t.append(check("less_negative_more_silence", sil_ratio_40,
               sil_ratio_60 - 0.05, 1.0))

# -80 dB (more negative) detects LESS silence than -60 dB  
t.append(check("more_negative_less_silence", sil_ratio_80,
               0.0, sil_ratio_60 + 0.05))

# Ordering: -40 >= -60 >= -80
ordering = float((sil_ratio_40 >= sil_ratio_60) and (sil_ratio_60 >= sil_ratio_80))
t.append(check("threshold_ordering_correct", ordering, 0.99, 1.01))
 
run_suite("threshold_sensitivity", t)


  ──────────────────────────────────────────────────────────────────────────
  Pure silence  (all zeros)
  ──────────────────────────────────────────────────────────────────────────
  Feature                                Actual   Expected Range         Status
  ──────────────────────────────────────────────────────────────────────────
  silence_ratio_100pct                 1.000000   [0.9900, 1.0100]   ✅ PASS
  silence_run_count_one                1.000000   [0.5000, 1.5000]   ✅ PASS
  silence_total_dur_full               1.927256   [1.9000, 2.1000]   ✅ PASS
  low_energy_ratio_100pct              1.000000   [0.9900, 1.0100]   ✅ PASS

  ✅ pure_silence (4/4 passed)

  ──────────────────────────────────────────────────────────────────────────
  Constant amplitude  (DC, no silence)
  ──────────────────────────────────────────────────────────────────────────
  Feature                                Actual   Expected Range         Status
  ─────────────────────────────────────────────────

In [50]:
# ═══════════════════════════════════════════════════════════════════════════
#  Summary
# ══════════════════════════════════════════════════════════════════════════════
total_pass = sum(p for p, _ in results.values())
total_all  = sum(t for _, t in results.values())
 
print("\n")
print("=" * WIDTH)
print("  TIMEFEATURES SILENCE STRUCTURE TESTBED REPORT")
print("=" * WIDTH)
print(f"  Total: {total_all} | PASS: {total_pass} | FAIL: {total_all - total_pass}")
print("=" * WIDTH)
for name, (p, tot) in results.items():
    icon = "✅" if p == tot else "❌"
    print(f"  {icon} {name:<35} {p}/{tot}")
print("=" * WIDTH)
 
if total_pass == total_all:
    print("\n🎉 All tests passed!")
else:
    print(f"\n⚠️  {total_all - total_pass} test(s) failed — review implementations")



  TIMEFEATURES SILENCE STRUCTURE TESTBED REPORT
  Total: 33 | PASS: 33 | FAIL: 0
  ✅ pure_silence                        4/4
  ✅ constant_amplitude                  4/4
  ✅ continuous_sine                     2/2
  ✅ silence_padded                      4/4
  ✅ alternating_silence                 4/4
  ✅ exponential_decay                   3/3
  ✅ step_amplitude                      2/2
  ✅ impulse_train                       3/3
  ✅ speech_like_pauses                  4/4
  ✅ threshold_sensitivity               3/3

🎉 All tests passed!


In [51]:
# ══════════════════════════════════════════════════════════════════════════════
#  Signal generators
# ══════════════════════════════════════════════════════════════════════════════
 
def make_silence(duration=3.0):
    """
    Pure silence.
    loudness: very low (-80 dB)
    energy: 0
    speechiness: low (no activity)
    danceability: 0 (no rhythm)
    """
    return np.zeros(int(duration * SR))
 
 
def make_loud_constant(duration=3.0, amplitude=0.9):
    """
    Loud constant tone.
    loudness: high (near 0 dB)
    energy: high (but no dynamics)
    danceability: low (no rhythm)
    """
    t = np.arange(int(duration * SR)) / SR
    return amplitude * np.sin(2 * np.pi * 440 * t)
 
 
def make_quiet_ambient(duration=3.0, amplitude=0.05):
    """
    Very quiet ambient noise.
    loudness: very low
    energy: very low
    acousticness: high (soft, dynamic)
    """
    rng = np.random.default_rng(42)
    noise = amplitude * rng.standard_normal(int(duration * SR))
    # Apply low-pass filter for ambient feel
    from scipy.signal import butter, filtfilt
    b, a = butter(4, 500 / (SR / 2), btype='low')
    return filtfilt(b, a, noise)
 
 
def make_speech_like(duration=4.0, n_utterances=6, amplitude=0.5):
    """
    Speech-like pattern: voiced bursts with pauses.
    speechiness: very high
    instrumentalness: very low
    silence_ratio: moderate (pauses)
    """
    blocks = []
    utterance_dur = 0.4
    pause_dur = 0.25
    
    for i in range(n_utterances):
        # Voiced segment (noisy with pitch variation)
        n_utt = int(utterance_dur * SR)
        rng = np.random.default_rng(42 + i)
        
        # Pitch-modulated noise (rough voice simulation)
        t = np.arange(n_utt) / SR
        pitch = 150 + 50 * np.sin(2 * np.pi * 2 * t)  # Varying pitch
        voiced = amplitude * np.sin(2 * np.pi * pitch * t)
        noise = 0.3 * rng.standard_normal(n_utt)
        utterance = voiced + noise
        
        blocks.append(utterance)
        
        # Pause
        if i < n_utterances - 1:
            blocks.append(np.zeros(int(pause_dur * SR)))
    
    return np.concatenate(blocks)
 
 
def make_electronic_dance(duration=4.0, bpm=128, amplitude=0.7):
    """
    Electronic dance music pattern: steady kick drum, hi-hats.
    danceability: very high
    acousticness: very low
    energy: high
    pulse_clarity: high
    """
    n = int(duration * SR)
    signal = np.zeros(n)
    
    # Kick drum on beats
    beat_period = 60.0 / bpm
    kick_positions = np.arange(0, duration, beat_period)
    
    for pos in kick_positions:
        idx = int(pos * SR)
        if idx + 1000 < n:
            # Exponential decay kick
            decay = amplitude * np.exp(-np.linspace(0, 8, 1000))
            kick = decay * np.sin(2 * np.pi * 60 * np.linspace(0, 1000/SR, 1000))
            signal[idx:idx+1000] += kick
    
    # Hi-hats on off-beats
    hihat_positions = np.arange(beat_period/2, duration, beat_period)
    for pos in hihat_positions:
        idx = int(pos * SR)
        if idx + 200 < n:
            # Short noise burst
            rng = np.random.default_rng(int(pos * 1000))
            hihat = 0.3 * amplitude * rng.standard_normal(200)
            hihat *= np.exp(-np.linspace(0, 10, 200))
            signal[idx:idx+200] += hihat
    
    return signal
 
 
def make_acoustic_guitar(duration=3.0, amplitude=0.6):
    """
    Acoustic guitar strums with dynamics.
    acousticness: very high
    energy: moderate
    dynamic_range: high
    transient_ratio: high
    """
    n = int(duration * SR)
    signal = np.zeros(n)
    
    # Strum positions (irregular timing)
    strum_times = [0.0, 0.6, 1.1, 1.8, 2.4]
    
    for i, t_strum in enumerate(strum_times):
        if t_strum >= duration:
            break
        
        idx = int(t_strum * SR)
        strum_len = 8000  # ~0.36 sec decay
        
        if idx + strum_len < n:
            # Multi-frequency pluck (guitar harmonics)
            t = np.arange(strum_len) / SR
            fundamental = 196  # G3
            
            # Harmonics with varying amplitudes
            tone = (
                1.0 * np.sin(2 * np.pi * fundamental * t) +
                0.5 * np.sin(2 * np.pi * 2 * fundamental * t) +
                0.3 * np.sin(2 * np.pi * 3 * fundamental * t)
            )
            
            # Exponential decay
            envelope = np.exp(-2.0 * t)
            
            # Vary amplitude per strum
            amp_var = 0.6 + 0.4 * (i % 3) / 2.0
            strum = amp_var * amplitude * tone * envelope
            
            signal[idx:idx+strum_len] += strum
    
    return signal
 
 
def make_steady_metronome(duration=3.0, bpm=120, amplitude=0.5):
    """
    Perfect metronome clicks.
    danceability: high (perfect rhythm)
    pulse_clarity: very high
    rhythmic_stability: very high
    """
    n = int(duration * SR)
    signal = np.zeros(n)
    
    beat_period = 60.0 / bpm
    click_positions = np.arange(0, duration, beat_period)
    
    for pos in click_positions:
        idx = int(pos * SR)
        if idx + 100 < n:
            # Sharp click (short decay)
            decay = amplitude * np.exp(-np.linspace(0, 15, 100))
            click = decay * np.sin(2 * np.pi * 1000 * np.linspace(0, 100/SR, 100))
            signal[idx:idx+100] += click
    
    return signal
 
 
def make_rubato_classical(duration=4.0, amplitude=0.5):
    """
    Classical piece with tempo variations (rubato).
    liveness: high (tempo fluctuations)
    rhythmic_stability: low
    danceability: low
    """
    n = int(duration * SR)
    signal = np.zeros(n)
    
    # Notes with irregular timing (rubato)
    note_times = [0.0, 0.4, 0.9, 1.5, 2.0, 2.3, 2.8, 3.4]
    frequencies = [262, 294, 330, 349, 392, 440, 494, 523]  # C major scale
    
    for t_note, freq in zip(note_times, frequencies):
        if t_note >= duration:
            break
        
        idx = int(t_note * SR)
        note_len = 6000
        
        if idx + note_len < n:
            t = np.arange(note_len) / SR
            
            # Piano-like tone
            tone = np.sin(2 * np.pi * freq * t)
            envelope = np.exp(-1.5 * t)
            
            note = amplitude * tone * envelope
            signal[idx:idx+note_len] += note
    
    return signal
 
 
def make_live_recording(duration=4.0, amplitude=0.5):
    """
    Simulated live recording: music + audience noise + tempo drift.
    liveness: very high
    silence_ratio: low (audience noise)
    rhythmic_stability: moderate-low
    """
    # Base rhythm with slight tempo variations
    n = int(duration * SR)
    signal = np.zeros(n)
    
    # Drifting tempo: 115 → 125 BPM
    bpm_start = 115
    bpm_end = 125
    
    t_current = 0.0
    beat_count = 0
    
    while t_current < duration:
        # Current BPM (linear drift)
        progress = t_current / duration
        current_bpm = bpm_start + (bpm_end - bpm_start) * progress
        beat_period = 60.0 / current_bpm
        
        idx = int(t_current * SR)
        if idx + 500 < n:
            # Beat sound
            decay = amplitude * np.exp(-np.linspace(0, 8, 500))
            beat = decay * np.sin(2 * np.pi * 80 * np.linspace(0, 500/SR, 500))
            signal[idx:idx+500] += beat
        
        t_current += beat_period
        beat_count += 1
    
    # Add ambient audience noise
    rng = np.random.default_rng(99)
    audience_noise = 0.1 * amplitude * rng.standard_normal(n)
    signal += audience_noise
    
    return signal
 
 
def make_compressed_pop(duration=3.0, amplitude=0.8):
    """
    Heavily compressed pop music simulation.
    energy: very high
    dynamic_range: very low
    loudness: high
    """
    n = int(duration * SR)
    t = np.arange(n) / SR
    
    # Multiple instruments at similar levels (compressed)
    bass = 0.3 * np.sin(2 * np.pi * 60 * t)
    melody = 0.3 * np.sin(2 * np.pi * 440 * t + np.sin(2 * np.pi * 2 * t))
    harmony = 0.2 * np.sin(2 * np.pi * 550 * t)
    
    # Combine and apply soft clipping (compression)
    signal = amplitude * (bass + melody + harmony)
    signal = np.tanh(signal * 2.0) / 2.0  # Soft clipping
    
    return signal

In [52]:
# ══════════════════════════════════════════════════════════════════════════════
#  Test runner
# ══════════════════════════════════════════════════════════════════════════════
 
PASS_MARK = "✅ PASS"
FAIL_MARK = "❌ FAIL"
WIDTH     = 74
 
 
def check(name, actual, lo, hi):
    ok = lo <= actual <= hi
    status = PASS_MARK if ok else FAIL_MARK
    print(f"  {name:<32} {actual:>12.6f}   [{lo:.4f}, {hi:.4f}]   {status}")
    return ok
 
 
def section(title):
    print(f"\n  {'─'*WIDTH}")
    print(f"  {title}")
    print(f"  {'─'*WIDTH}")
    print(f"  {'Feature':<32} {'Actual':>12}   {'Expected Range':<20}   Status")
    print(f"  {'─'*WIDTH}")
 
 
results = {}
 
 
def run_suite(name, tests):
    passed = sum(tests)
    total  = len(tests)
    results[name] = (passed, total)
    icon = "✅" if passed == total else "❌"
    print(f"\n  {icon} {name} ({passed}/{total} passed)")

In [55]:
# ══════════════════════════════════════════════════════════════════════════════
#  Individual test suites
# ══════════════════════════════════════════════════════════════════════════════
 
# ─────────────────────────────────────────────────────────────────────────────
#  1. Silence — minimal everything
# ─────────────────────────────────────────────────────────────────────────────
section("Silence  (all zeros)")
y_sil = make_silence(duration=3.0)
sig_sil = AudioSignal(signal=y_sil, N=N, H=H)
feat_sil = TimeFeatures(sig_sil)
 
t = []
t.append(check("loudness_dB_very_low",       feat_sil.loudness_dB(),       -80.0, -70.0))
t.append(check("loudness_norm_zero",         feat_sil.loudness_norm(),     0.0, 0.05))
t.append(check("energy_zero",                feat_sil.energy_partial(),    0.0, 0.05))
t.append(check("speechiness_low", feat_sil.speechiness(), 0.0, 0.25))
t.append(check("danceability_zero", feat_sil.danceability_partial(), 0.0, 0.25))
t.append(check("instrumentalness_high", feat_sil.instrumentalness(), 0.75, 1.0))
 
run_suite("silence", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  2. Loud constant tone — high loudness, low dynamics
# ─────────────────────────────────────────────────────────────────────────────
section("Loud constant tone  (high amplitude sine)")
y_loud = make_loud_constant(duration=3.0, amplitude=0.9)
sig_loud = AudioSignal(signal=y_loud, N=N, H=H)
feat_loud = TimeFeatures(sig_loud)
 
t = []
t.append(check("loudness_dB_high",           feat_loud.loudness_dB(),       -10.0, 0.0))
t.append(check("loudness_norm_high",         feat_loud.loudness_norm(),     0.80, 1.0))
t.append(check("energy_low_no_dynamics",     feat_loud.energy_partial(),    0.0, 0.40))
t.append(check("danceability_low",           feat_loud.danceability_partial(), 0.0, 0.30))
t.append(check("acousticness_low",           feat_loud.acousticness_partial(), 0.0, 0.30))
 
run_suite("loud_constant", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  3. Quiet ambient — low loudness, high acousticness
# ─────────────────────────────────────────────────────────────────────────────
section("Quiet ambient  (soft filtered noise)")
y_ambient = make_quiet_ambient(duration=3.0, amplitude=0.05)
sig_ambient = AudioSignal(signal=y_ambient, N=N, H=H)
feat_ambient = TimeFeatures(sig_ambient)
 
t = []
t.append(check("loudness_dB_very_low",       feat_ambient.loudness_dB(),       -60.0, -30.0))
t.append(check("loudness_norm_low",          feat_ambient.loudness_norm(),     0.0, 0.35))
t.append(check("energy_very_low",            feat_ambient.energy_partial(),    0.0, 0.25))
t.append(check("acousticness_moderate", feat_ambient.acousticness_partial(), 0.20, 1.0))
t.append(check("danceability_very_low", feat_ambient.danceability_partial(), 0.0, 0.30))
 
run_suite("quiet_ambient", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  4. Speech-like — high speechiness, low instrumentalness
# ─────────────────────────────────────────────────────────────────────────────
section("Speech-like  (voiced bursts with pauses)")
y_speech = make_speech_like(duration=4.0, n_utterances=6, amplitude=0.5)
sig_speech = AudioSignal(signal=y_speech, N=N, H=H)
feat_speech = TimeFeatures(sig_speech)
 
t = []
t.append(check("speechiness_high",           feat_speech.speechiness(),       0.40, 0.90))
t.append(check("instrumentalness_low",       feat_speech.instrumentalness(),  0.10, 0.60))
# Complementarity check
speech_val = feat_speech.speechiness()
instr_val = feat_speech.instrumentalness()
complementarity = abs((speech_val + instr_val) - 1.0)
t.append(check("speech_instr_complementary", complementarity, 0.0, 0.05))
 
run_suite("speech_like", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  5. Electronic dance — high danceability, low acousticness
# ─────────────────────────────────────────────────────────────────────────────
section("Electronic dance  (steady kick + hi-hats)")
y_dance = make_electronic_dance(duration=4.0, bpm=128, amplitude=0.7)
sig_dance = AudioSignal(signal=y_dance, N=N, H=H)
feat_dance = TimeFeatures(sig_dance)
 
t = []
t.append(check("danceability_high",          feat_dance.danceability_partial(), 0.50, 1.0))
t.append(check("energy_high",                feat_dance.energy_partial(),    0.40, 1.0))
t.append(check("acousticness_low", feat_dance.acousticness_partial(), 0.0, 0.85))
t.append(check("tempo_near_128",             feat_dance.tempo_partial(),     110.0, 145.0))
 
run_suite("electronic_dance", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  6. Acoustic guitar — high acousticness, moderate energy
# ─────────────────────────────────────────────────────────────────────────────
section("Acoustic guitar  (strums with dynamics)")
y_guitar = make_acoustic_guitar(duration=3.0, amplitude=0.6)
sig_guitar = AudioSignal(signal=y_guitar, N=N, H=H)
feat_guitar = TimeFeatures(sig_guitar)
 
t = []
t.append(check("acousticness_very_high",     feat_guitar.acousticness_partial(), 0.60, 1.0))
t.append(check("energy_moderate",            feat_guitar.energy_partial(), 0.0, 0.65))
t.append(check("danceability_low",           feat_guitar.danceability_partial(), 0.0, 0.45))
t.append(check("instrumentalness_high",      feat_guitar.instrumentalness(),  0.60, 1.0))
 
run_suite("acoustic_guitar", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  7. Steady metronome — very high danceability, pulse clarity
# ─────────────────────────────────────────────────────────────────────────────
section("Steady metronome  (perfect 120 BPM)")
y_metro = make_steady_metronome(duration=3.0, bpm=120, amplitude=0.5)
sig_metro = AudioSignal(signal=y_metro, N=N, H=H)
feat_metro = TimeFeatures(sig_metro)
 
t = []
t.append(check("danceability_very_high",     feat_metro.danceability_partial(), 0.65, 1.0))
t.append(check("tempo_120_bpm",              feat_metro.tempo_partial(),     110.0, 130.0))
# Time signature should detect some regularity
time_sig, conf = feat_metro.time_signature_partial()
t.append(check("time_sig_confidence",        conf,                           0.20, 1.0))
 
run_suite("steady_metronome", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  8. Rubato classical — low rhythmic stability, low danceability
# ─────────────────────────────────────────────────────────────────────────────
section("Rubato classical  (tempo variations)")
y_rubato = make_rubato_classical(duration=4.0, amplitude=0.5)
sig_rubato = AudioSignal(signal=y_rubato, N=N, H=H)
feat_rubato = TimeFeatures(sig_rubato)
 
t = []
t.append(check("danceability_low",       feat_rubato.danceability_partial(), 0.0, 0.45))
t.append(check("liveness_moderate_high", feat_rubato.liveness_partial(),  0.05, 0.50))
t.append(check("acousticness_high",          feat_rubato.acousticness_partial(), 0.45, 1.0))
 
run_suite("rubato_classical", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  9. Live recording — high liveness, tempo drift
# ─────────────────────────────────────────────────────────────────────────────
section("Live recording  (audience noise + tempo drift)")
y_live = make_live_recording(duration=4.0, amplitude=0.5)
sig_live = AudioSignal(signal=y_live, N=N, H=H)
feat_live = TimeFeatures(sig_live)
 
t = []
t.append(check("liveness_high",          feat_live.liveness_partial(),    0.01, 0.60))
t.append(check("danceability_moderate",  feat_live.danceability_partial(),   0.20, 0.75))
t.append(check("energy_moderate_high",       feat_live.energy_partial(),    0.30, 0.85))
 
run_suite("live_recording", t)
 
 
# ─────────────────────────────────────────────────────────────────────────────
#  10. Compressed pop — high energy, low dynamic range
# ─────────────────────────────────────────────────────────────────────────────
section("Compressed pop  (soft clipping, low DR)")
y_pop = make_compressed_pop(duration=3.0, amplitude=0.8)
sig_pop = AudioSignal(signal=y_pop, N=N, H=H)
feat_pop = TimeFeatures(sig_pop)
 
t = []
t.append(check("loudness_high",              feat_pop.loudness_dB(),       -15.0, 0.0))
t.append(check("energy_high",                feat_pop.energy_partial(),    0.45, 1.0))
t.append(check("acousticness_low",           feat_pop.acousticness_partial(), 0.0, 0.35))
 
run_suite("compressed_pop", t)


  ──────────────────────────────────────────────────────────────────────────
  Silence  (all zeros)
  ──────────────────────────────────────────────────────────────────────────
  Feature                                Actual   Expected Range         Status
  ──────────────────────────────────────────────────────────────────────────
  loudness_dB_very_low               -80.000000   [-80.0000, -70.0000]   ✅ PASS
  loudness_norm_zero                   0.000000   [0.0000, 0.0500]   ✅ PASS
  energy_zero                          0.000000   [0.0000, 0.0500]   ✅ PASS
  speechiness_low                      0.000000   [0.0000, 0.2500]   ✅ PASS
  danceability_zero                    0.250000   [0.0000, 0.2500]   ✅ PASS
  instrumentalness_high                1.000000   [0.7500, 1.0000]   ✅ PASS

  ✅ silence (6/6 passed)

  ──────────────────────────────────────────────────────────────────────────
  Loud constant tone  (high amplitude sine)
  ───────────────────────────────────────────────────────

In [56]:
# ═══════════════════════════════════════════════════════════════════════════
#  Summary
# ══════════════════════════════════════════════════════════════════════════════
total_pass = sum(p for p, _ in results.values())
total_all  = sum(t for _, t in results.values())
 
print("\n")
print("=" * WIDTH)
print("  TIMEFEATURES SPOTIFY-LIKE FEATURES TESTBED REPORT")
print("=" * WIDTH)
print(f"  Total: {total_all} | PASS: {total_pass} | FAIL: {total_all - total_pass}")
print("=" * WIDTH)
for name, (p, tot) in results.items():
    icon = "✅" if p == tot else "❌"
    print(f"  {icon} {name:<35} {p}/{tot}")
print("=" * WIDTH)
 
if total_pass == total_all:
    print("\n🎉 All tests passed!")
else:
    print(f"\n⚠️  {total_all - total_pass} test(s) failed — review feature implementations or ranges")



  TIMEFEATURES SPOTIFY-LIKE FEATURES TESTBED REPORT
  Total: 39 | PASS: 39 | FAIL: 0
  ✅ silence                             6/6
  ✅ loud_constant                       5/5
  ✅ quiet_ambient                       5/5
  ✅ speech_like                         3/3
  ✅ electronic_dance                    4/4
  ✅ acoustic_guitar                     4/4
  ✅ steady_metronome                    3/3
  ✅ rubato_classical                    3/3
  ✅ live_recording                      3/3
  ✅ compressed_pop                      3/3

🎉 All tests passed!


In [ ]:
# Real Tracks (TimeFeatures)
track_dir = "dataset/songs"

